# Position

Lets see if we can do interesting small boards

In [ ]:
#| default_exp game/mechanics/graph

In [ ]:
import sys
import math
from fastcore.basics import patch
from fasthtml.common import *
from fasthtml.jupyter import *
import pandas as pd
import numpy as np

In [ ]:
from HexMagic.primitives import MapPath, MapSize, MapRect, MapCord 
from HexMagic.primitives import HexGrid, HexPosition ,  HexRegion, GosperCurve, windy_edge , unique_windy_edge

import numpy as np

from HexMagic.terrain import Terrain
from HexMagic.climate import ClimatePreset, Climate, TerraDemo, TerrainFactory
from HexMagic.geology import Geology, DrainageBasins, SoilSystem
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.overlay import TerrainDisplay, CreamOverlay, RiverOverlay, ClimateOverlay

In [ ]:
from HexMagic.game.globals import appRoutes,  webMe, globalStore, ensure_user,new_game_page, create_game, create_world, invalidate_cache, showUsers, logging
from HexMagic.game.data import Settlement, Kingdom, Piece, TradeRoute, GameBoard, ActiveGame, CountryFlag, _map_point
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList, Squad
from HexMagic.game.data import RANK_ORDER 
from HexMagic.game.piece import CountryFlag, DiagramGlyphs,PieceBoard, PieceStep, _move_cost,PieceBoardPlan
from HexMagic.game.piece import SquadPlanOverlay, PieceList, FormationType, SurroundMode, TroopPath, _facing_toward
from HexMagic.game.piece import GroupedPieceList, SquadPlanOverlay, piece_plan_overlay, SquadSymbolOverlay #, SquadVisionOverlay
from HexMagic.game.piece import FormationType, _formation_offsets

from HexMagic.game.piece import pieces_center, StatBar, SquadVisionOverlay, GameParts,SettlementOverlay, KingdomNamesOverlay, PieceOverlay

In [ ]:
from HexMagic.game.food import FoodOverlay, FoodYield, PieceOverlay, PieceDataOverlay, PieceArrowOverlay, SquadSymbolOverlay, food_triage, pre_march_distribute, compose_callbacks

In [ ]:
from HexMagic.game.mechanics.simulator import TurnReport, SimEventType, FoodSimulator, squad_chart, SquadFoodTrailOverlay


In [ ]:
import pyomo.environ as pyo
import optuna

## Substainabilty

In [ ]:
def sustain_cost(hex_idx, grid, food_tiers, elevations,
                 queen_diet=1.9, pawn_diet=1.9, pawn_harvest=2):
    """How many pawns needed to sustain a queen camped at hex_idx.
    Returns int (1-99) or -1 for unsustainable.
    """
    # Best food tier in ring-1 (where pawns would sit)
    best_tier = 0
    for nb in grid.neighborsOf(hex_idx):
        if 0 <= nb < len(food_tiers) and elevations[nb] > 0:
            best_tier = max(best_tier, int(food_tiers[nb]))
    
    # Net food a pawn can export per turn (harvest every other turn)
    net = 0.5 * pawn_harvest * best_tier - pawn_diet
    
    if net <= 0:
        return -1  # unsustainable
    
    return math.ceil(queen_diet / net)


In [ ]:
def SustainOverlay(queen, pawn_template=None, **kw) -> OverlaySpec:
    """Color hexes by number of pawns needed to sustain the queen there.
    
    Green (1-2), Yellow (3-4), Orange (5-6), Red (7+), Gray (impossible).
    """
    COLORS = {
        1: '#27ae60', 2: '#2ecc71',  # great
        3: '#f1c40f', 4: '#f39c12',  # expensive
        5: '#e67e22', 6: '#d35400',  # costly
    }
    DEFAULT = '#e74c3c'  # 7+
    IMPOSSIBLE = '#444444'
    
    def render(ctx):
        grid = ctx.grid
        fy = FoodYield(ctx.terrain, ctx.basins)
        fy.compute()
        food_tiers = fy.tiers
        elevs = ctx.terrain.elevations
        
        parts = []
        for i in range(len(elevs)):
            if elevs[i] <= 0:
                continue
            cost = sustain_cost(i, grid, food_tiers, elevs,
                                queen_diet=queen.diet)
            if cost == -1:
                color = IMPOSSIBLE
            else:
                color = COLORS.get(cost, DEFAULT)
            
            h = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            parts.append(
                f'<polygon points="{pts}" fill="{color}" opacity="0.3"/>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("sustain_cost", render, priority=45)


## Camp Assign

In [ ]:
def _solver_ok(result):
    return (result.solver.status == pyo.SolverStatus.ok and
            result.solver.termination_condition == pyo.TerminationCondition.optimal)


In [ ]:
@patch
def camp_assign(squad:Squad, target_hex, grid, elevations, food_tiers,
                countries=None, max_ring=3,
                elevation_mult=0.005) -> dict:
    """Facility-location MIP: station feeders on fertile tiles to cover all demand.
    
    Minimizes total combat value assigned to feeding duty,
    keeping the best fighters free.
    """
    alive = squad.alive
    if not alive: return {}
    
    # ── Candidate tiles (facilities) ──
    tiles = []  # (hex_idx, tier)
    occupied = {target_hex}
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target_hex)
            if idx < 0 or idx in grid.invalidRegion: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier < 1: continue
            tiles.append((idx, tier))
    
    if not tiles: return {}
    
    N = len(alive)      # pieces (potential feeders AND consumers)
    T = len(tiles)       # candidate tiles
    
    # ── Precompute: which (tile, facing) can see which piece locations ──
    # For each tile, compute sight cone facing target_hex
    visible = np.zeros((T, N), dtype=bool)
    for t, (tidx, _) in enumerate(tiles):
        facing = _facing_toward(grid, tidx, target_hex)
        facing_dir = HexPosition.directions()[facing]
        elev = max(0, elevations[tidx])
        eff_sight = 3 + elev * elevation_mult
        fov = set()
        for hp in HexPosition.origin().field_of_view(facing_dir, int(eff_sight)):
            fov_idx = grid.hexposition_to_index(hp, tidx)
            if fov_idx >= 0: fov.add(fov_idx)
        # Target hex and ring-1 around it are where consumers will be
        for c, piece in enumerate(alive):
            # Consumer location = target_hex (they're camped)
            if target_hex in fov:
                visible[t, c] = True
    
    # ── Production capacity per (feeder, tile) ──
    # feed_turns = move_strength - 1 harvest actions per turn
    production = np.zeros((N, T))
    for f, piece in enumerate(alive):
        for t, (tidx, tier) in enumerate(tiles):
            production[f, t] = piece.harvest_strength * tier
    
    diet = np.array([p.diet for p in alive])
    combat = np.array([getattr(p, 'attack_strength', 0) for p in alive])
    
    # ── Pyomo model ──
    m = pyo.ConcreteModel()
    m.F = pyo.RangeSet(0, N - 1)    # feeders (any piece)
    m.T = pyo.RangeSet(0, T - 1)    # tiles
    m.C = pyo.RangeSet(0, N - 1)    # consumers (same pieces)
    
    # assign[f,t] = 1 if piece f is stationed on tile t
    m.assign = pyo.Var(m.F, m.T, domain=pyo.Binary)
    
    # flow[f,t,c] = food from feeder f on tile t to consumer c
    m.flow = pyo.Var(m.F, m.T, m.C, domain=pyo.NonNegativeReals)
    
    # ── Objective: minimize combat value of assigned feeders ──
    m.obj = pyo.Objective(
        expr=sum(combat[f] * m.assign[f, t] for f in m.F for t in m.T),
        sense=pyo.minimize)
    
    # ── Each piece assigned to at most 1 tile ──
    m.one_tile = pyo.Constraint(m.F, rule=lambda m, f:
        sum(m.assign[f, t] for t in m.T) <= 1)
    
    # ── Each tile has at most 1 piece ──
    m.one_per_tile = pyo.Constraint(m.T, rule=lambda m, t:
        sum(m.assign[f, t] for f in m.F) <= 1)
    
    # ── Flow only if assigned AND visible ──
    M = max(diet) * 2  # big-M
    def flow_limit(m, f, t, c):
        if not visible[t, c]:
            return m.flow[f, t, c] == 0
        return m.flow[f, t, c] <= M * m.assign[f, t]
    m.flow_cap = pyo.Constraint(m.F, m.T, m.C, rule=flow_limit)
    
    # ── Production capacity per feeder-tile ──
    m.prod_cap = pyo.Constraint(m.F, m.T, rule=lambda m, f, t:
        sum(m.flow[f, t, c] for c in m.C) <= production[f, t] * m.assign[f, t])
    
    # ── Every consumer's diet must be met ──
    m.demand = pyo.Constraint(m.C, rule=lambda m, c:
        sum(m.flow[f, t, c] for f in m.F for t in m.T) >= diet[c])
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    
    if not _solver_ok(result):
        return {}
    
    # ── Extract assignments ──
    orders = {}
    for f in m.F:
        piece = alive[f]
        assigned_tile = None
        for t in m.T:
            if pyo.value(m.assign[f, t]) > 0.5:
                assigned_tile = t
                break
        
        if assigned_tile is None:
            # Free piece — hold position at target
            orders[piece.id] = InstructionList(
                [Instruction.PAUSE.value], cursor=0, patrol=True)
        else:
            tidx, _ = tiles[assigned_tile]
            facing = _facing_toward(grid, tidx, target_hex)
            
            tp = piece.pathfind(tidx, grid, elevations, countries)
            rules = tp.to_rules(grid, piece.facing) if tp else []
            
            # Rotate to face target
            cur = piece.facing if not rules else _facing_toward(
                grid, tp.hexes[-2] if len(tp.hexes) >= 2 else piece.location, tidx)
            diff = (facing - cur) % 6
            if diff <= 3: rules.extend([Instruction.ROT_R.value] * diff)
            else: rules.extend([Instruction.ROT_L.value] * (6 - diff))
            
            rules.extend([Instruction.HARVEST.value, Instruction.GIVE.value])
            orders[piece.id] = InstructionList(rules, cursor=0, patrol=True)
    
        # ── Extract assignments (add to existing camp_assign) ──
    feeder_ids = set()
    tile_for = {}  # piece_id → hex_idx
    
    for f in m.F:
        piece = alive[f]
        for t in m.T:
            if pyo.value(m.assign[f, t]) > 0.5:
                feeder_ids.add(piece.id)
                tile_for[piece.id] = tiles[t][0]
                break
    
    orders['feeder_ids'] = feeder_ids
    orders['tile_for'] = tile_for
    orders['total_production'] = sum(
        alive[f].harvest_strength * tiles[t][1]
        for f in m.F for t in m.T
        if pyo.value(m.assign[f, t]) > 0.5)
    
    return orders



In [ ]:
@patch
def supply_assign(squad:Squad, target_hex, grid, elevations, food_tiers,
                countries=None, max_ring=3):
    """Facility-location MIP: pick who feeds, on which tile, minimizing combat loss."""
    alive = squad.alive
    if not alive: return {'feasible': False}
    
    # ── Candidate tiles around camp center ──
    tiles = []
    for ring in range(1, max_ring + 1):
        for hp in HexPosition.origin().ring(ring):
            idx = grid.hexposition_to_index(hp, target_hex)
            if idx < 0 or idx in grid.invalidRegion: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            tier = int(food_tiers[idx]) if 0 <= idx < len(food_tiers) else 0
            if tier < 1: continue
            tiles.append((idx, tier))
    
    if not tiles: return {'feasible': False}
    
    N, T = len(alive), len(tiles)
    prod = np.zeros((N, T))
    for f, p in enumerate(alive):
        for t, (_, tier) in enumerate(tiles):
            prod[f, t] = p.harvest_strength * tier
    
    total_diet = sum(p.diet for p in alive)
    combat = np.array([getattr(p, 'attack_strength', 0) for p in alive])
    
    m = pyo.ConcreteModel()
    m.F = pyo.RangeSet(0, N - 1)
    m.T = pyo.RangeSet(0, T - 1)
    m.x = pyo.Var(m.F, m.T, domain=pyo.Binary)
    
    m.obj = pyo.Objective(
        expr=sum(combat[f] * m.x[f, t] for f in m.F for t in m.T),
        sense=pyo.minimize)
    
    m.one_tile = pyo.Constraint(m.F, rule=lambda m, f:
        sum(m.x[f, t] for t in m.T) <= 1)
    m.one_per = pyo.Constraint(m.T, rule=lambda m, t:
        sum(m.x[f, t] for f in m.F) <= 1)
    m.demand = pyo.Constraint(
        expr=sum(prod[f, t] * m.x[f, t] for f in m.F for t in m.T) >= total_diet)
    
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    if not _solver_ok(result):
        return {'feasible': False}
    
    feeder_ids, tile_for = set(), {}
    total_prod = 0
    for f in m.F:
        for t in m.T:
            if pyo.value(m.x[f, t]) > 0.5:
                p = alive[f]
                feeder_ids.add(p.id)
                tile_for[p.id] = tiles[t][0]
                total_prod += prod[f, t]
    
    return dict(feasible=True, feeder_ids=feeder_ids, tile_for=tile_for,
                total_production=total_prod, total_diet=total_diet,
                net_surplus=total_prod - total_diet, tiles=tiles)


In [ ]:
@patch
def camp_economics(squad:Squad, camp_result, tiles, food_tiers):
    """Compute per-turn surplus from a camp_assign solution.
    
    Returns (total_production, total_diet, net_surplus, deficit_to_fill)
    """
    alive = squad.alive
    total_diet = sum(p.diet for p in alive)
    
    # Sum production from assigned feeders
    total_prod = 0
    for p in alive:
        tile_idx = camp_result.get('tile_for', {}).get(p.id)
        if tile_idx is not None:
            tier = int(food_tiers[tile_idx])
            total_prod += p.harvest_strength * tier
    
    deficit = sum(max(0, p.food_capacity - p.food) for p in alive)
    net = total_prod - total_diet
    
    return dict(production=total_prod, diet=total_diet,
                net_surplus=net, deficit=deficit,
                turns_to_fill=math.ceil(deficit / net) if net > 0 else float('inf'))


In [ ]:
@patch
def apply_food_profile(self: GameParts, soil_scale: float = 1.0):
    """Set food stats on all pieces based on balanced profile.
    
    Design intent:
      - Queen is expensive to feed (3.5× pawn), can't self-sustain
      - Pawns are field workers: decent harvest, small pack
      - Bishops are supply wagons: bad harvest, huge capacity
      - Knights are fast but hungry scouts
      - Rooks are heavy but carry well
      - King eats like a queen (royalty!)
    """
    PROFILES = {
        #                  diet  harvest  capacity
        PieceType.PAWN:   (2.0,  2.0,     12),
        PieceType.BISHOP: (2.5,  1.0,    100),
        PieceType.KNIGHT: (3.0,  1.5,     20),
        PieceType.ROOK:   (3.5,  1.0,     40),
        PieceType.QUEEN:  (7.0,  0.5,     30),
        PieceType.KING:   (6.0,  0.5,     25),
    }
    
    count = {t.name: 0 for t in PieceType}
    for country in self.board.kingdoms:
        for city in country.settlements:
            for piece in city.citizens:
                profile = PROFILES.get(piece.piece_type)
                if profile is None:
                    continue
                piece.diet, piece.harvest_strength, piece.food_capacity = profile
                piece.food = piece.food_capacity  # start full
                count[piece.piece_type.name] += 1
    
    # Apply soil scaling if needed

    fy = FoodYield(self.terr, self.basin)
    if soil_scale != 1.0:
        fy.soil_mult = [s * soil_scale for s in fy.soil_mult]
    fy.compute()

    
    print(f"Food profile applied:")
    for ptype, (d, h, c) in PROFILES.items():
        n = count.get(ptype.name, 0)
        net_t5 = 0.5 * h * 5 - d  # net at tier 5
        label = "✅" if net_t5 > 0 else "❌"
        print(f"  {ptype.name:8s} ×{n:2d}: diet={d:.1f} harvest={h:.1f} "
              f"cap={c:3d}  net@tier5={net_t5:+.1f} {label}")
    
    queen_d = PROFILES[PieceType.QUEEN][0]
    pawn_net = 0.5 * PROFILES[PieceType.PAWN][1] * 5 - PROFILES[PieceType.PAWN][0]
    bish_gap = PROFILES[PieceType.BISHOP][2] / (PROFILES[PieceType.BISHOP][0] + queen_d)
    print(f"\n  Pawns to sustain queen @tier5: {math.ceil(queen_d / max(pawn_net, 0.01))}")
    print(f"  Bishop gap crossing: ~{int(bish_gap)} barren hexes")
    return fy


In [ ]:
@patch
def playability_check(self: GameBoard, max_pawns=5):
    """Check if all settlements are connected via sustainable corridors.
    
    Returns dict with:
      - connected: bool
      - coverage: float (fraction of land that's sustainable)
      - corridors: dict of settlement pairs → path exists
      - sustainable_count / land_count: raw numbers
    """
    fy = FoodYield(self.terrain, self.basins); fy.compute()
    food_tiers = fy.tiers
    elevs = self.terrain.elevations
    grid = self.grid
    
    # Build sustainable hex set
    sustainable = set()
    for i in range(len(elevs)):
        if elevs[i] <= 0:
            continue
        cost = sustain_cost(i, grid, food_tiers, elevs)
        if 0 < cost <= max_pawns:
            sustainable.add(i)
    
    land = sum(1 for e in elevs if e > 0)
    coverage = len(sustainable) / max(land, 1)
    
    # BFS connectivity between settlements using only sustainable hexes
    def bfs_connected(start, end):
        if start not in sustainable or end not in sustainable:
            return False
        visited = {start}
        queue = [start]
        while queue:
            current = queue.pop(0)
            if current == end:
                return True
            for nb in grid.neighborsOf(current):
                if nb in sustainable and nb not in visited:
                    visited.add(nb)
                    queue.append(nb)
        return False
    
    settlements = self.settlements
    corridors = {}
    locs = [s.location for s in settlements]
    all_connected = True
    for i in range(len(locs)):
        for j in range(i+1, len(locs)):
            ok = bfs_connected(locs[i], locs[j])
            corridors[(i, j)] = ok
            if not ok:
                all_connected = False
    
    return {
        'connected': all_connected,
        'coverage': coverage,
        'corridors': corridors,
        'sustainable_count': len(sustainable),
        'land_count': land,
    }


In [ ]:
def _find_hungry(alive, grid, piece, safety_turns=2):
    """Find the most desperate piece, sorted by urgency."""
    hungry = []
    for p in alive:
        if p is piece: continue
        if p.health <= 0: continue
        turns_left = p.food / max(p.diet, 0.1)
        if turns_left < safety_turns * 3:  # worth helping
            hungry.append((p, turns_left))
    hungry.sort(key=lambda h: h[1])  # most desperate first
    return hungry


def _adjacent(grid, loc_a, loc_b):
    """True if same hex or neighbors."""
    if loc_a == loc_b: return True
    return loc_b in grid.neighborsOf(loc_a)


def _donor_surplus(piece, safety_turns=2):
    """How much food this piece can spare."""
    keep = piece.diet * safety_turns
    return max(0, piece.food - keep)


def food_redistribute(report: TurnReport):
    """Squad-level food sharing — three tiers of urgency.
    
    Tier 1: Direct transfer (adjacent, no instruction cost)
    Tier 2: Rotate toward hungry (prepend ROT, then GIVE next turn)
    Tier 3: Move toward hungry (prepend FORWARD steps toward them)
    
    Only donors with surplus above their own safety margin participate.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    elevations = report.elevations
    
    # ── Tier 1: Direct transfers (free, every turn) ──
    by_need = sorted(alive, key=lambda p: p.food / max(p.diet, 0.1))
    
    for receiver in by_need:
        need = receiver.food_capacity - receiver.food
        if need < 0.1: continue
        
        for donor in alive:
            if donor is receiver: continue
            surplus = _donor_surplus(donor)
            if surplus < 0.1: continue
            if not _adjacent(grid, donor.location, receiver.location):
                continue
            
            transfer = min(need, surplus)
            donor.food -= transfer
            receiver.food += transfer
            need -= transfer
            if need < 0.1: break
    
    # ── Tier 2 & 3: Only for pieces with surplus that aren't feeding ──
    # Check if anyone is still desperate after direct transfers
    desperate = [p for p in alive 
                 if p.food / max(p.diet, 0.1) < 3.0]
    if not desperate: return
    
    most_desperate = min(desperate, key=lambda p: p.food / max(p.diet, 0.1))
    
    for donor in alive:
        if donor is most_desperate: continue
        surplus = _donor_surplus(donor)
        if surplus < 1.0: continue
        if _adjacent(grid, donor.location, most_desperate.location):
            continue  # already handled in tier 1
        
        # ── Tier 2: Rotate to face the hungry piece ──
        target_facing = _facing_toward(grid, donor.location, 
                                        most_desperate.location)
        if donor.facing != target_facing:
            diff = (target_facing - donor.facing) % 6
            if diff <= 3:
                rot = Instruction.ROT_R.value
            else:
                rot = Instruction.ROT_L.value
            
            cursor = donor.instructions.cursor
            # Only insert if not already rotating toward them
            rules = donor.instructions.rules
            if cursor < len(rules) and rules[cursor] in (
                    Instruction.ROT_R.value, Instruction.ROT_L.value):
                continue  # already rotating, don't stack
            
            donor.instructions.rules.insert(cursor, rot)
            continue  # rotate this turn, give next turn
        
        # ── Tier 3: Facing correct but not adjacent — move toward ──
        # Only move if the receiver is truly desperate (< 2 turns)
        recv_turns = most_desperate.food / max(most_desperate.diet, 0.1)
        if recv_turns > 2.0: continue
        
        # Check if forward movement gets us closer
        hp_donor = grid.index_to_hexposition(donor.location)
        hp_recv  = grid.index_to_hexposition(most_desperate.location)
        dist_now = hp_donor.distance(hp_recv)
        
        # Peek at hex in front
        fwd_hp = hp_donor.neighbor(donor.facing)
        fwd_idx = grid.hexposition_to_index(fwd_hp, donor.location)
        
        if (fwd_idx >= 0 and 
            fwd_idx not in grid.invalidRegion and
            elevations[fwd_idx] > 0):
            
            fwd_dist = fwd_hp.distance(hp_recv)
            if fwd_dist < dist_now:
                cursor = donor.instructions.cursor
                rules = donor.instructions.rules
                if cursor < len(rules) and rules[cursor] == Instruction.FORWARD.value:
                    continue  # already moving
                
                donor.instructions.rules.insert(cursor, Instruction.FORWARD.value)


def food_harvest_check(report: TurnReport):
    """Prepend HARVEST when a piece is on a food tile and hungry.
    
    Capped at 1 insert per turn — won't corrupt instruction lists.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    
    for piece in alive:
        loc = piece.location
        if loc is None or loc < 0: continue
        if piece.food >= piece.food_capacity * 0.8: continue
        
        tier = int(report.food_tiers[loc]) if 0 <= loc < len(report.food_tiers) else 0
        if tier < 1: continue
        
        rules = piece.instructions.rules
        cursor = piece.instructions.cursor
        if cursor < len(rules) and rules[cursor] == Instruction.HARVEST.value:
            continue
        
        piece.instructions.rules.insert(cursor, Instruction.HARVEST.value)


def food_triage_v2(report: TurnReport):
    """Combined: harvest opportunities, then redistribute/rescue."""
    food_harvest_check(report)
    food_redistribute(report)


In [ ]:
def food_redistribute_adjacent(report: TurnReport):
    """Tier 1: Direct adjacent transfers. No instructions touched.
    
    Safe to call every turn, even during march.
    Moves food from anyone with surplus to the most desperate neighbor.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    
    # Most desperate first
    by_need = sorted(alive, key=lambda p: p.food / max(p.diet, 0.1))
    
    for receiver in by_need:
        need = receiver.food_capacity - receiver.food
        if need < 0.1: continue
        
        # Donors: adjacent pieces with surplus above safety margin
        for donor in alive:
            if donor is receiver: continue
            surplus = _donor_surplus(donor)
            if surplus < 0.1: continue
            if not _adjacent(grid, donor.location, receiver.location):
                continue
            
            transfer = min(need, surplus)
            donor.food -= transfer
            receiver.food += transfer
            need -= transfer
            if need < 0.1: break


def food_redistribute_rescue(report: TurnReport):
    """Tiers 2-3: Rotate/move toward hungry. Edits instructions.
    
    Only call during CAMP or emergency. Will disrupt march orders.
    """
    alive = [p for p in report.squad.pieces if p.health > 0]
    if len(alive) < 2: return
    grid = report.grid
    elevations = report.elevations
    
    # Anyone still desperate after tier-1?
    desperate = [p for p in alive 
                 if p.food / max(p.diet, 0.1) < 3.0]
    if not desperate: return
    
    most_desperate = min(desperate, key=lambda p: p.food / max(p.diet, 0.1))
    
    for donor in alive:
        if donor is most_desperate: continue
        surplus = _donor_surplus(donor)
        if surplus < 1.0: continue
        if _adjacent(grid, donor.location, most_desperate.location):
            continue  # tier 1 already handled
        
        # ── Tier 2: Rotate toward hungry ──
        target_facing = _facing_toward(grid, donor.location, 
                                        most_desperate.location)
        if donor.facing != target_facing:
            diff = (target_facing - donor.facing) % 6
            rot = Instruction.ROT_R.value if diff <= 3 else Instruction.ROT_L.value
            
            cursor = donor.instructions.cursor
            rules = donor.instructions.rules
            if cursor < len(rules) and rules[cursor] in (
                    Instruction.ROT_R.value, Instruction.ROT_L.value):
                continue
            
            rules.insert(cursor, rot)
            continue
        
        # ── Tier 3: Move toward hungry (only if truly dying) ──
        recv_turns = most_desperate.food / max(most_desperate.diet, 0.1)
        if recv_turns > 2.0: continue
        
        hp_donor = grid.index_to_hexposition(donor.location)
        hp_recv  = grid.index_to_hexposition(most_desperate.location)
        dist_now = hp_donor.distance(hp_recv)
        
        fwd_hp = hp_donor.neighbor(donor.facing)
        fwd_idx = grid.hexposition_to_index(fwd_hp, donor.location)
        
        if (fwd_idx >= 0 and 
            fwd_idx not in grid.invalidRegion and
            elevations[fwd_idx] > 0):
            
            fwd_dist = fwd_hp.distance(hp_recv)
            if fwd_dist < dist_now:
                cursor = donor.instructions.cursor
                rules = donor.instructions.rules
                if cursor < len(rules) and rules[cursor] == Instruction.FORWARD.value:
                    continue
                rules.insert(cursor, Instruction.FORWARD.value)


## Max Flow

## flow

So I am wondering if we need something like feedcover for smart_placement. There is a general graph layout/network flow problem here. we clearly want to put the pawns on the high yield hexes. then we need edges to both the stores {bishops) and the sinks (the queens). and then we need a set of edges from the stores to the sinks. we want to max the flow of the graph. sight cones are how we can have a one way arc in the network flow. is this something for pyomo. Is there away of doing optimal layout given a set of hexes?

Yes — this is a beautiful fit for Pyomo. The key reframe: **hex grid = directed graph, pieces = node capacities, GIVE = arc flow, sight cones = directed arcs**.

```python
def optimal_camp_layout(squad, center, grid, elevations, food_tiers,
                        countries=None, max_ring=3, elevation_mult=0.005):
    """Network-flow camp placement MIP.
    
    The hex neighborhood is a directed graph:
      - Nodes: candidate hexes (occupied by pieces)
      - Arcs: adjacency edges (or sight-cone edges for longer range)
      - Source nodes: pawns on food hexes (production = harvest × tier)
      - Relay nodes: bishops (high throughput, buffer capacity)
      - Sink nodes: queen/king (high diet, low harvest)
    
    Flow conservation at each occupied hex:
      production + inflow ≥ consumption + outflow
    
    Placement decides WHO goes WHERE.
    Flow decides HOW MUCH food moves along each arc.
    Objective: maximize net surplus (total prod - total diet),
               tiebreak by keeping high-combat pieces off food duty.
    """
    alive = squad.alive
    N = len(alive)
    
    # ── Candidate hexes (ring 0..max_ring around center) ──
    cands = []
    seen = set()
    for ring in range(0, max_ring + 1):
        ring_hps = [HexPosition.origin()] if ring == 0 else HexPosition.origin().ring(ring)
        for hp in ring_hps:
            idx = grid.hexposition_to_index(hp, center)
            if idx < 0 or idx in grid.invalidRegion or idx in seen: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            seen.add(idx)
            cands.append(idx)
    
    H = len(cands)
    tier = [int(food_tiers[c]) if 0 <= c < len(food_tiers) else 0
            for c in cands]
    hex_to_local = {c: i for i, c in enumerate(cands)}
    
    # ── Directed arcs: adjacency-based ──
    # (For sight-cone arcs, expand this with FOV precomputation)
    arcs = []
    for i in range(H):
        for nb in grid.neighborsOf(cands[i]):
            if nb in hex_to_local:
                j = hex_to_local[nb]
                arcs.append((i, j))
    
    # Piece parameters
    diet_p = [p.diet for p in alive]
    harv_p = [p.harvest_strength for p in alive]
    combat_p = [getattr(p, 'attack_strength', 0) for p in alive]
    
    # ── Pyomo model ──
    m = pyo.ConcreteModel()
    m.P = pyo.RangeSet(0, N - 1)
    m.H = pyo.RangeSet(0, H - 1)
    m.A = pyo.Set(initialize=arcs)
    
    # ── Decision variables ──
    m.place = pyo.Var(m.P, m.H, domain=pyo.Binary)    # piece p on hex h
    m.flow  = pyo.Var(m.A, domain=pyo.NonNegativeReals) # food along arc
    
    # ── Placement constraints ──
    m.one_hex = pyo.Constraint(m.P, rule=lambda m, p:
        sum(m.place[p, h] for h in m.H) == 1)
    
    m.one_piece = pyo.Constraint(m.H, rule=lambda m, h:
        sum(m.place[p, h] for p in m.P) <= 1)
    
    # ── Production / consumption at each hex ──
    def _prod(m, h):
        """Harvest output: Σ_p place[p,h] × harv_p × tier[h]"""
        return sum(m.place[p, h] * harv_p[p] * tier[h] for p in m.P)
    
    def _cons(m, h):
        """Diet demand: Σ_p place[p,h] × diet_p"""
        return sum(m.place[p, h] * diet_p[p] for p in m.P)
    
    # ── Flow conservation at every hex ──
    # production + inflow ≥ consumption + outflow
    def balance(m, h):
        inflow  = sum(m.flow[j, h] for (j, hh) in arcs if hh == h)
        outflow = sum(m.flow[h, j] for (hh, j) in arcs if hh == h)
        return _prod(m, h) + inflow >= _cons(m, h) + outflow
    m.balance = pyo.Constraint(m.H, rule=balance)
    
    # ── Arc capacity: can only send from an occupied hex ──
    M_big = max(harv_p) * max(tier + [1]) * 2
    m.arc_send = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, i] for p in m.P))
    m.arc_recv = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, j] for p in m.P))
    
    # ── Objective ──
    # Primary: maximize total surplus (production - consumption)
    # Secondary: minimize combat value on food hexes (keep fighters free)
    α = 100  # surplus is much more important than combat tiebreak
    
    # A piece is "on food duty" if on a hex with tier > 0
    food_duty_combat = sum(
        combat_p[p] * m.place[p, h] * (1 if tier[h] > 0 else 0)
        for p in range(N) for h in range(H))
    
    m.obj = pyo.Objective(
        expr=α * (sum(_prod(m, h) for h in m.H) 
                - sum(_cons(m, h) for h in m.H))
             - food_duty_combat,
        sense=pyo.maximize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, tee=False)
    if not _solver_ok(result):
        return None
    
    # ── Extract results ──
    placements = {}  # piece → hex_idx
    for p in m.P:
        for h in m.H:
            if pyo.value(m.place[p, h]) > 0.5:
                placements[alive[p].id] = cands[h]
                break
    
    flows = {}  # (hex_i, hex_j) → flow amount
    for (i, j) in arcs:
        v = pyo.value(m.flow[i, j])
        if v > 0.01:
            flows[(cands[i], cands[j])] = round(v, 2)
    
    total_prod = sum(pyo.value(_prod(m, h)) for h in m.H)
    total_cons = sum(pyo.value(_cons(m, h)) for h in m.H)
    
    return {
        'placements': placements,
        'flows': flows,
        'production': total_prod,
        'consumption': total_cons,
        'surplus': total_prod - total_cons,
        'cands': cands,
        'tiers': tier,
    }
```

The network structure:

```
     [Pawn A]──harvest──→ food
         │
    adj arc (GIVE)
         │
         ▼
    [Bishop]──buffer──→ receives, eats 2.5, forwards rest
         │
    adj arc (GIVE)
         │
         ▼
     [Queen]──sink──→ eats 7.0
```

Flow conservation does all the work:
- **Pawn on tier-5 hex**: produces `2.0 × 5 = 10`, eats 2.0, can export 8.0 along arcs
- **Bishop on tier-0 hex**: produces 0, receives inflow, eats 2.5, forwards the rest
- **Queen**: produces ~0, receives inflow, eats 7.0 — must have enough inflow to survive

The MIP simultaneously decides:
1. **Where** each piece goes (which hex)
2. **How much** food flows along each edge
3. **Who** is on food duty vs free (combat tiebreak)

To add **sight cones as directed arcs** (longer range than adjacency):

```python
# ── Extended arcs: sight-cone based ──
for i in range(H):
    for d in range(6):
        facing_dir = HexPosition.directions()[d]
        hp_i = grid.index_to_hexposition(cands[i])
        sight = 3 + max(0, elevations[cands[i]]) * elevation_mult
        for hp_vis in hp_i.field_of_view(facing_dir, int(sight)):
            j_idx = grid.hexposition_to_index(hp_vis, cands[i])
            if j_idx in hex_to_local:
                j = hex_to_local[j_idx]
                if j != i:
                    arcs.append((i, j))  # directed: i can see j
```

This makes the graph **directed** — a pawn facing north can send food 3 hexes north, but not south. The MIP picks placements that create efficient food pipelines. Bishops become relay stations at graph junctions.

For the overlay, the `flows` dict is perfect — draw arrows between hexes with thickness proportional to flow amount. The player sees the food supply chain on the map.

In [ ]:
def optimal_camp_layout(squad, center, grid, elevations, food_tiers,
                        countries=None, max_ring=3, elevation_mult=0.005):
    """Network-flow camp placement MIP.
    
    The hex neighborhood is a directed graph:
      - Nodes: candidate hexes (occupied by pieces)
      - Arcs: adjacency edges (or sight-cone edges for longer range)
      - Source nodes: pawns on food hexes (production = harvest × tier)
      - Relay nodes: bishops (high throughput, buffer capacity)
      - Sink nodes: queen/king (high diet, low harvest)
    
    Flow conservation at each occupied hex:
      production + inflow ≥ consumption + outflow
    
    Placement decides WHO goes WHERE.
    Flow decides HOW MUCH food moves along each arc.
    Objective: maximize net surplus (total prod - total diet),
               tiebreak by keeping high-combat pieces off food duty.
    """
    alive = squad.alive
    N = len(alive)
    
    # ── Candidate hexes (ring 0..max_ring around center) ──
    cands = []
    seen = set()
    for ring in range(0, max_ring + 1):
        ring_hps = [HexPosition.origin()] if ring == 0 else HexPosition.origin().ring(ring)
        for hp in ring_hps:
            idx = grid.hexposition_to_index(hp, center)
            if idx < 0 or idx in grid.invalidRegion or idx in seen: continue
            if elevations[idx] <= 0: continue
            if countries is not None and countries[idx] < 0: continue
            seen.add(idx)
            cands.append(idx)
    
    H = len(cands)
    tier = [int(food_tiers[c]) if 0 <= c < len(food_tiers) else 0
            for c in cands]
    hex_to_local = {c: i for i, c in enumerate(cands)}
    
    # ── Directed arcs: adjacency-based ──
    # (For sight-cone arcs, expand this with FOV precomputation)
    arcs = []
    for i in range(H):
        for nb in grid.neighborsOf(cands[i]):
            if nb in hex_to_local:
                j = hex_to_local[nb]
                arcs.append((i, j))
    
    # Piece parameters
    diet_p = [p.diet for p in alive]
    harv_p = [p.harvest_strength for p in alive]
    combat_p = [getattr(p, 'attack_strength', 0) for p in alive]
    
    # ── Pyomo model ──
    m = pyo.ConcreteModel()
    m.P = pyo.RangeSet(0, N - 1)
    m.H = pyo.RangeSet(0, H - 1)
    m.A = pyo.Set(initialize=arcs)
    
    # ── Decision variables ──
    m.place = pyo.Var(m.P, m.H, domain=pyo.Binary)    # piece p on hex h
    m.flow  = pyo.Var(m.A, domain=pyo.NonNegativeReals) # food along arc
    
    # ── Placement constraints ──
    m.one_hex = pyo.Constraint(m.P, rule=lambda m, p:
        sum(m.place[p, h] for h in m.H) == 1)
    
    m.one_piece = pyo.Constraint(m.H, rule=lambda m, h:
        sum(m.place[p, h] for p in m.P) <= 1)
    
    # ── Production / consumption at each hex ──
    def _prod(m, h):
        """Harvest output: Σ_p place[p,h] × harv_p × tier[h]"""
        return sum(m.place[p, h] * harv_p[p] * tier[h] for p in m.P)
    
    def _cons(m, h):
        """Diet demand: Σ_p place[p,h] × diet_p"""
        return sum(m.place[p, h] * diet_p[p] for p in m.P)
    
    # ── Flow conservation at every hex ──
    # production + inflow ≥ consumption + outflow
    def balance(m, h):
        inflow  = sum(m.flow[j, h] for (j, hh) in arcs if hh == h)
        outflow = sum(m.flow[h, j] for (hh, j) in arcs if hh == h)
        return _prod(m, h) + inflow >= _cons(m, h) + outflow
    m.balance = pyo.Constraint(m.H, rule=balance)
    
    # ── Arc capacity: can only send from an occupied hex ──
    M_big = max(harv_p) * max(tier + [1]) * 2
    m.arc_send = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, i] for p in m.P))
    m.arc_recv = pyo.Constraint(m.A, rule=lambda m, i, j:
        m.flow[i, j] <= M_big * sum(m.place[p, j] for p in m.P))
    
    # ── Objective ──
    # Primary: maximize total surplus (production - consumption)
    # Secondary: minimize combat value on food hexes (keep fighters free)
    α = 100  # surplus is much more important than combat tiebreak
    
    # A piece is "on food duty" if on a hex with tier > 0
    food_duty_combat = sum(
        combat_p[p] * m.place[p, h] * (1 if tier[h] > 0 else 0)
        for p in range(N) for h in range(H))
    
    m.obj = pyo.Objective(
        expr=α * (sum(_prod(m, h) for h in m.H) 
                - sum(_cons(m, h) for h in m.H))
             - food_duty_combat,
        sense=pyo.maximize)
    
    # ── Solve ──
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, tee=False)
    if not _solver_ok(result):
        return None
    
    # ── Extract results ──
    placements = {}  # piece → hex_idx
    for p in m.P:
        for h in m.H:
            if pyo.value(m.place[p, h]) > 0.5:
                placements[alive[p].id] = cands[h]
                break
    
    flows = {}  # (hex_i, hex_j) → flow amount
    for (i, j) in arcs:
        v = pyo.value(m.flow[i, j])
        if v > 0.01:
            flows[(cands[i], cands[j])] = round(v, 2)
    
    total_prod = sum(pyo.value(_prod(m, h)) for h in m.H)
    total_cons = sum(pyo.value(_cons(m, h)) for h in m.H)
    
    return {
        'placements': placements,
        'flows': flows,
        'production': total_prod,
        'consumption': total_cons,
        'surplus': total_prod - total_cons,
        'cands': cands,
        'tiers': tier,
    }


In [ ]:
def smart_placement_v2(settlement, squad, terrain, food_tiers, grid,
                       max_ring=10, camp_ring=3, top_k=5):
    """Place entire squad using optimal_camp_layout network-flow MIP.
    
    Phase 1: Score candidate centers cheaply (sum of nearby food tiers)
    Phase 2: Run full MIP on top-k candidates
    Phase 3: Apply best layout
    
    Returns the best result dict from optimal_camp_layout, or None.
    """
    center = settlement.location
    elevs = terrain.elevations
    
    # ── Phase 1: Cheap scoring of candidate centers ──
    candidates = []
    for hp in HexPosition.origin().spiral(max_ring):
        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0 or elevs[idx] <= 0: continue
        
        # Quick heuristic: sum food tiers in the camp neighborhood
        score = 0
        land_count = 0
        for r in range(0, camp_ring + 1):
            ring_hps = [HexPosition.origin()] if r == 0 else HexPosition.origin().ring(r)
            for nhp in ring_hps:
                nidx = grid.hexposition_to_index(nhp, origin_index=idx)
                if nidx < 0 or elevs[nidx] <= 0: continue
                land_count += 1
                score += int(food_tiers[nidx]) if 0 <= nidx < len(food_tiers) else 0
        
        # Need enough land hexes for the squad
        if land_count >= len(squad.alive):
            candidates.append((idx, score))
    
    if not candidates:
        return None
    
    # Top-k by cheap score
    candidates.sort(key=lambda x: -x[1])
    shortlist = candidates[:top_k]
    
    # ── Phase 2: Full MIP on shortlist ──
    best_result = None
    best_surplus = -float('inf')
    best_center = center
    
    for cand_idx, cheap_score in shortlist:
        result = optimal_camp_layout(
            squad, cand_idx, grid, elevs, food_tiers,
            max_ring=camp_ring)
        
        if result is None: continue
        if result['surplus'] > best_surplus:
            best_surplus = result['surplus']
            best_result = result
            best_center = cand_idx
    
    if best_result is None:
        return None
    
    # ── Phase 3: Apply placements + facings ──
    for piece in squad.alive:
        hex_idx = best_result['placements'].get(piece.id)
        if hex_idx is not None:
            piece.location = hex_idx
            # Face toward camp center (for GIVE arcs)
            if hex_idx != best_center:
                piece.facing = _facing_toward(grid, hex_idx, best_center)
    
    best_result['center'] = best_center
    print(f"Smart placement v2: center={best_center}, "
          f"surplus={best_surplus:.1f} "
          f"(prod={best_result['production']:.1f}, "
          f"cons={best_result['consumption']:.1f}), "
          f"checked {len(shortlist)} candidates")
    
    return best_result


In [ ]:
@patch
def loadGeology(self: GameParts, geo: Geology):
    self.terr = geo.terrain
    self.grid = self.terr.hexGrid
    self.builder = self.grid.builder

    self.board = GameBoard(self.terr, 2)
    self.basin = geo.basins

    self.country = self.board.kingdoms[0]
    self.home = self.country.settlements[0]
    self.squads = Squad.squads(6, self.country.flag)
    self.queenGuard = self.squads[0]
    for pType in [PieceType.PAWN, PieceType.PAWN, PieceType.PAWN, PieceType.PAWN,
                  PieceType.PAWN, PieceType.BISHOP, PieceType.BISHOP, PieceType.QUEEN]:
        piece = self.home.recruit(pType, flag=self.queenGuard.flag)
        self.queenGuard.pieces.append(piece)

    # Initial placement so pieces have locations
    self.home.place_squad_spiral(self.queenGuard, self.terr)
    self.queen = self.queenGuard.by_rank()[0]

    # Set diet/harvest/capacity on all pieces
    self.apply_food_profile()

    # Compute food tiers, then re-place using sustainability logic
    fy = FoodYield(self.terr, self.basin)
    fy.compute()
    best_cost = smart_placement_v2(self.home, self.queenGuard, self.terr, fy.tiers, self.grid)
    print(f"Smart placement: queen at {self.queen.location}, sustain cost = {best_cost} pawns")


given optimal_camp_layout can we have a version of smart_placement that would use it,

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Absolutely — the key win is that `optimal_camp_layout` jointly places *everyone* (queen included), so you don't need the separate greedy queen scan anymore. The natural approach is a two-phase design: cheap heuristic to shortlist centers, then full MIP on the finalists.

```python
def smart_placement_v2(settlement, squad, terrain, food_tiers, grid,
                       max_ring=10, camp_ring=3, top_k=5):
    """Place entire squad using optimal_camp_layout network-flow MIP.
    
    Phase 1: Score candidate centers cheaply (sum of nearby food tiers)
    Phase 2: Run full MIP on top-k candidates
    Phase 3: Apply best layout
    
    Returns the best result dict from optimal_camp_layout, or None.
    """
    center = settlement.location
    elevs = terrain.elevations
    
    # ── Phase 1: Cheap scoring of candidate centers ──
    candidates = []
    for hp in HexPosition.origin().spiral(max_ring):
        idx = grid.hexposition_to_index(hp, origin_index=center)
        if idx < 0 or elevs[idx] <= 0: continue
        
        # Quick heuristic: sum food tiers in the camp neighborhood
        score = 0
        land_count = 0
        for r in range(0, camp_ring + 1):
            ring_hps = [HexPosition.origin()] if r == 0 else HexPosition.origin().ring(r)
            for nhp in ring_hps:
                nidx = grid.hexposition_to_index(nhp, origin_index=idx)
                if nidx < 0 or elevs[nidx] <= 0: continue
                land_count += 1
                score += int(food_tiers[nidx]) if 0 <= nidx < len(food_tiers) else 0
        
        # Need enough land hexes for the squad
        if land_count >= len(squad.alive):
            candidates.append((idx, score))
    
    if not candidates:
        return None
    
    # Top-k by cheap score
    candidates.sort(key=lambda x: -x[1])
    shortlist = candidates[:top_k]
    
    # ── Phase 2: Full MIP on shortlist ──
    best_result = None
    best_surplus = -float('inf')
    best_center = center
    
    for cand_idx, cheap_score in shortlist:
        result = optimal_camp_layout(
            squad, cand_idx, grid, elevs, food_tiers,
            max_ring=camp_ring)
        
        if result is None: continue
        if result['surplus'] > best_surplus:
            best_surplus = result['surplus']
            best_result = result
            best_center = cand_idx
    
    if best_result is None:
        return None
    
    # ── Phase 3: Apply placements + facings ──
    for piece in squad.alive:
        hex_idx = best_result['placements'].get(piece.id)
        if hex_idx is not None:
            piece.location = hex_idx
            # Face toward camp center (for GIVE arcs)
            if hex_idx != best_center:
                piece.facing = _facing_toward(grid, hex_idx, best_center)
    
    best_result['center'] = best_center
    print(f"Smart placement v2: center={best_center}, "
          f"surplus={best_surplus:.1f} "
          f"(prod={best_result['production']:.1f}, "
          f"cons={best_result['consumption']:.1f}), "
          f"checked {len(shortlist)} candidates")
    
    return best_result
```

The structure:

1. **Phase 1** is O(candidates × ring²) — just summing tier values, very fast. This filters the search space from hundreds of hexes down to `top_k`.

2. **Phase 2** runs `top_k` MIPs (default 5). Each is small (8 pieces × ~30 hexes), so HiGHS should solve them in milliseconds each.

3. **Phase 3** applies the winning layout and orients everyone inward so their GIVE arcs point toward the center of the food network.

The big difference from `smart_placement` v1: the queen's hex isn't chosen independently — the MIP might put her on a tier-0 hex surrounded by high-tier pawns if that maximizes the *network* surplus. The flow conservation constraint handles the relay logic (pawn → bishop → queen) automatically.

You could also wire this into `loadGeology` as a drop-in replacement — just swap `smart_placement(...)` for `smart_placement_v2(...)` and return the surplus from the result dict.

### Show Demo

In [ ]:
showDemo = True

In [ ]:
showDemo = False

In [ ]:
myStuff = GameParts()

In [ ]:
def rebootWorld():

    world = TerrainFactory.create_world(
        bounds= MapRect(MapCord(0, 0), MapSize(300, 300)),
        preset='temperate',
        name='Maiden Lane',
        radius=15,
        lon_span=10.0,
        num_plates=8,
        subdivisions=3,
        ocean_fraction=0.3,
        oceanic_sides=['N'],  # Ocean on east and west
        terrain_age='young',  # Sharp, dramatic features
        formation_type='ridge',  # Creates ridge formations
        elevation_scale=1.5,  # Exaggerate the heights
        erosion_age=0.1,  # Minimal erosion for sharp peaks
        num_lakes=0,
        seed=23,
        debug=True
    )

    sampleMap = world.terrain
    sampleMap.repalette()

    for i in range(len(sampleMap.hexGrid.hexes)):
        sampleMap.hexGrid.hexes[i].label = str(i)

    myStuff.loadGeology(world)
    myStuff.grid.adjustRadius(40)

    return myStuff

In [ ]:
rebootWorld()

In [ ]:
for i in range(len(myStuff.grid.hexes)):
    myStuff.grid.hexes[i].label = str(i)

In [ ]:
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    SquadVisionOverlay([myStuff.queenGuard]),
    SustainOverlay(queen=myStuff.queen),
    RiverOverlay(max_width=5),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
    )

In [ ]:
GroupedPieceList(myStuff.queenGuard, grid= myStuff.grid)

In [ ]:
# Run the simulator with food_triage so the squad stops to feed
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
food_tiers = fy.tiers

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=food_tiers,
    pieces=myStuff.queenGuard.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[myStuff.queenGuard],
    callbacks={myStuff.queenGuard.id: food_triage_v2},
)

sim.run(num_turns=40)
sim.summary()


so we want to move pieces on the board. for a given squad we are going to have a set of pieces that are moving and a set of pieces that are feeding. another option is to have two squads which might be easier. lets call it squad M for move and squad S for share. with their sight cones squad S can share its food across a certain number of hexes and can rotate to share others. much as we have optimal camp layout we can have a sinks and sources. a move piece can move along these squares as a sink. to advance we alternate squads so that each provides to each other.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

This is a beautiful leapfrog logistics pattern — essentially "bounding overwatch" but for food instead of fire. The two-squad split makes it much cleaner than trying to track feeder/mover roles within one squad.

The core loop:

```
Phase A:  S harvests & shares →  M advances through S's food shadow
Phase B:  M camps & harvests  →  S leapfrogs past M
Repeat
```

The key insight is that sight cones create a **food shadow** — the set of hexes a feeder can reach with GIVE. As M moves along a path, it passes through (or outside) those shadows. The question each phase answers is: *how far can M advance before it leaves S's food coverage?*

Here's the network structure for a single bound:

```
Turn 1:    [S1]──cone──→ [M at path[0]]  ✓ fed
Turn 2:    [S1]──cone──→ [M at path[1]]  ✓ still in cone
Turn 3:    [S1]──cone──→ [M at path[2]]  ✗ out of range → S rotates
Turn 4:    [S1]──cone──→ [M at path[3]]  ✓ new facing covers it
Turn 5:    [S1]────✗────  [M at path[4]]  out of all cones → bound ends
```

The MIP for each bound would be:

- **Fixed**: march path (precomputed), S piece positions (from previous camp_layout)
- **Variables**: S facings per turn (can rotate once/turn), food flow arcs
- **Objective**: maximize advance distance (how many path hexes M can traverse while fed)
- **Output**: the "bound length" — how far M gets before roles swap

The two things to figure out:

1. **Bound planning MIP** — given S positions and a march path, what facings should S use each turn to maximize coverage? This is a multi-period flow problem where the sinks move.

2. **Leapfrog orchestrator** — alternates phases, runs camp_layout for the new feeder squad, computes the next bound, repeats until destination.

Want to start with the bound planner (how far can M go in one phase), or the orchestrator (the full A→B→A loop)?

does it make sense to have 3 groups? 1 for the queen and some bishops and 2 groups of pawns? ultimately these pawn groups create feed structures so perhaps the game is configuring pawn groups to let your big guns battle? So maybe even more pawn groups?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — this is the right strategic reframe. The queen group is the **payload**, pawn groups are **infrastructure**. The game becomes: *how do you configure your supply chain?*

Three groups is the natural minimum for continuous movement:

```
P1: harvesting (feeding Q as it passes)
 Q: advancing through P1's food shadow
P2: leapfrogging ahead, setting up next station
```

With only 2, the queen has to **stop** while the forward group sets up. With 3, there's always one group ahead preparing the next waypoint. It's the difference between stop-and-go and smooth advance.

But the really interesting design space opens with **more pawn groups**:

**Relay chain** — instead of leapfrogging, pawn groups form a static pipeline:
```
[P1]──cone──→[P2]──cone──→[P3]──cone──→[Q+Bishops]
  ↑ harvest      relay        relay        sink
```
Each group harvests *and* receives from behind, forwarding surplus to the next node. Bishops act as buffers at junctions. The queen never harvests — she just consumes the end of the pipeline.

**The player decisions become:**
- **Group count**: more groups = thinner each, but longer reach
- **Group size**: 2 pawns per group barely self-sustains, 3 creates surplus to export
- **Topology**: linear chain? hub-and-spoke around the queen? split to cover multiple food patches?
- **Route planning**: the march path is now constrained by *where pawn groups can set up fertile stations* — you're pathfinding through food, not just terrain

**The strategic tradeoff** is beautiful: every pawn on food duty is a pawn *not* fighting. So the question is always "what's the minimum supply chain that keeps my queen alive and moving?" A player who builds an elegant 3-group pipeline with 6 pawns keeps 4 fighters free. A sloppy player wastes 8 pawns on food and has nothing left for combat.

For the implementation, I'd suggest the pawn groups don't need to be formal `Squad` objects — they could be **roles within a squad**, tagged by assignment:

```python
@dataclass
class SupplyRole:
    PAYLOAD = 'payload'   # queen, bishops, fighters — never harvest
    STATION = 'station'   # pawn group currently camped & harvesting
    TRANSIT = 'transit'   # pawn group leapfrogging to next position

@dataclass  
class SupplyNode:
    """A feeding station along the march route."""
    pieces: list          # pawns assigned here
    hex_idx: int          # camp center
    coverage: set         # hexes this station can feed (sight cones)
    production: float     # food/turn output
    role: SupplyRole
```

Then the **march planner** becomes: given the route, place `SupplyNode`s along it such that their coverage zones tile the path, and the total production covers the payload's diet. That's another MIP — or even just a greedy sweep along the path.

Want to sketch the `SupplyNode` + march planner, or nail down the pawn group sizing math first (how many pawns per station to create exportable surplus at each tier)?

In [ ]:
@dataclass
class SupplyRole:
    PAYLOAD = 'payload'   # queen, bishops, fighters — never harvest
    STATION = 'station'   # pawn group currently camped & harvesting
    TRANSIT = 'transit'   # pawn group leapfrogging to next position

@dataclass  
class SupplyNode:
    """A feeding station along the march route."""
    pieces: list          # pawns assigned here
    hex_idx: int          # camp center
    coverage: set         # hexes this station can feed (sight cones)
    production: float     # food/turn output
    role: SupplyRole


So when I tried this before we had something that would try to schedule waypoints. and I think we could do something similar about placing supplynodes. below is a class and maybe these is a SupplyNode kind of version of this

```python
class LeapSchedule:
    """Optimal camping schedule for a squad march."""
    
    def __init__(self, squad, path, df, net_prod, total_diet, total_cap):
        self.squad = squad
        self.path = path          # TroopPath
        self.df = df              # pandas DataFrame
        self.net_prod = net_prod  # per-waypoint net production rates
        self.total_diet = total_diet
        self.total_cap = total_cap
    
    @property
    def total_camp(self): return int(self.df['camp_turns'].sum())
    
    @property
    def total_turns(self): 
        return self.total_camp + int(self.df['march_turns'].sum())
    
    @property
    def camp_stops(self): return self.df[self.df['camp_turns'] > 0]
    
    def __repr__(self):
        stops = len(self.camp_stops)
        return (f"LeapSchedule({len(self.df)} waypoints, "
                f"{stops} camps, {self.total_camp} camp turns, "
                f"~{self.total_turns} total)")
    
    @classmethod
    def plan(cls, squad, target, grid, elevations, food_tiers, 
            countries=None, min_food_turns=2.0, max_camp_per_stop=60,
            refine=True):
        """Two-pass schedule: cheap heuristic → refine camp stops with network-flow MIP."""
        leader = squad.by_rank()[0]
        alive = squad.alive
        
        lpath = leader.pathfind(target, grid, elevations, countries)
        if not lpath: return None
        
        W = len(lpath.hexes)
        total_diet = sum(p.diet for p in alive)
        total_cap = sum(p.food_capacity for p in alive)
        food0 = sum(p.food for p in alive)
        min_food = total_diet * min_food_turns
        
        # ── Pass 1: cheap net_prod estimates ──
        feeders = sorted([p for p in alive if p.harvest_strength > 0],
                        key=lambda p: -p.harvest_strength)
        
        net_prod = []
        for w, whex in enumerate(lpath.hexes):
            tiles = _food_tiles_at(whex, grid, elevations, food_tiers, countries)
            net_prod.append(_waypoint_net_prod(tiles, feeders, total_diet))
        
        march_turns = _estimate_march_turns(lpath, grid, elevations, alive)
        march_cost = [total_diet * mt for mt in march_turns]
        
        # ── Solve initial schedule ──
        sched_df = cls._solve_schedule(
            W, net_prod, march_cost, food0, min_food, total_cap, max_camp_per_stop)
        if sched_df is None: return None
        
        # ── Pass 2: refine camp stops with optimal_camp_layout ──
        if refine:
            refined = False
            for w in range(W):
                if sched_df[w]['camp_turns'] == 0: continue
                
                whex = lpath.hexes[w]
                layout = optimal_camp_layout(
                    squad, whex, grid, elevations, food_tiers,
                    countries=countries, max_ring=3)
                
                if layout is not None:
                    true_surplus = layout['surplus']
                    if abs(true_surplus - net_prod[w]) > 0.5:
                        net_prod[w] = true_surplus
                        refined = True
                        sched_df[w]['layout'] = layout  # stash for later use
            
            # Re-solve with corrected production rates
            if refined:
                sched_df = cls._solve_schedule(
                    W, net_prod, march_cost, food0, min_food, 
                    total_cap, max_camp_per_stop)
                if sched_df is None: return None
        
        # ── Build DataFrame ──
        rows = []
        for w in range(W):
            ct = sched_df[w]['camp_turns']
            af = sched_df[w]['arrival_food']
            df_val = sched_df[w]['departure_food']
            mt = march_turns[w] if w < W - 1 else 0
            rows.append({
                'waypoint': w,
                'hex': lpath.hexes[w],
                'camp_turns': ct,
                'march_turns': mt,
                'net_prod': net_prod[w],
                'arrival_food': af,
                'departure_food': df_val,
                'march_cost': march_cost[w] if w < W - 1 else 0,
                'food_pct': af / total_cap if total_cap > 0 else 0,
            })
        
        sched = cls(squad=squad, path=lpath,
                    df=pd.DataFrame(rows),
                    net_prod=net_prod,
                    total_diet=total_diet,
                    total_cap=total_cap)
        
        # Attach layouts for SquadMission to use at execution time
        sched.layouts = {w: sched_df[w].get('layout')
                        for w in range(W) if sched_df[w].get('layout')}
        
        total_mt = sum(march_turns)
        camps = sched.camp_stops
        tag = "✅" if not refined else "✅🔬"
        print(f"{tag} Schedule: {W} waypoints, {len(camps)} camps, "
            f"{sched.total_camp} camp + {total_mt} march = "
            f"~{sched.total_camp + total_mt} total turns")
        for _, s in camps.iterrows():
            layout_tag = " 📐" if int(s['waypoint']) in sched.layouts else ""
            print(f"   wp{int(s['waypoint'])} ⬡{int(s['hex'])}: "
                f"⛺{int(s['camp_turns'])}t (net {s['net_prod']:+.1f}/t)  "
                f"food {s['arrival_food']:.0f}→{s['departure_food']:.0f}{layout_tag}")
        
        return sched

    @classmethod
    def _solve_schedule(cls, W, net_prod, march_cost, food0, min_food, 
                        total_cap, max_camp_per_stop):
        """Inner MIP solve — reusable for both passes."""
        m = pyo.ConcreteModel()
        m.W = pyo.RangeSet(0, W - 1)
        
        m.camp = pyo.Var(m.W, domain=pyo.NonNegativeIntegers,
                        bounds=(0, max_camp_per_stop))
        m.A = pyo.Var(m.W, domain=pyo.NonNegativeReals, bounds=(0, total_cap))
        m.F = pyo.Var(m.W, domain=pyo.NonNegativeReals, bounds=(0, total_cap))
        
        m.obj = pyo.Objective(expr=sum(m.camp[w] for w in m.W), sense=pyo.minimize)
        
        m.a_init = pyo.Constraint(expr=m.A[0] == food0)
        m.f_camp = pyo.Constraint(m.W, rule=lambda m, w:
            m.F[w] == m.A[w] + m.camp[w] * net_prod[w])
        m.a_flow = pyo.Constraint(range(1, W), rule=lambda m, w:
            m.A[w] == m.F[w - 1] - march_cost[w - 1])
        m.survive = pyo.Constraint(m.W, rule=lambda m, w:
            m.A[w] >= min_food)
        m.capacity = pyo.Constraint(m.W, rule=lambda m, w:
            m.F[w] <= total_cap)
        
        for w in range(W):
            if net_prod[w] <= 0:
                m.camp[w].fix(0)
        
        solver = pyo.SolverFactory('appsi_highs')
        try:
            result = solver.solve(m, load_solutions=False)
            if not _solver_ok(result): return None
            m.solutions.load_from(result)
        except RuntimeError:
            return None
        
        # Return as list of dicts (mutable, can attach layout later)
        rows = []
        for w in range(W):
            rows.append({
                'camp_turns': int(round(pyo.value(m.camp[w]))),
                'arrival_food': pyo.value(m.A[w]),
                'departure_food': pyo.value(m.F[w]),
            })
        return rows




```

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — it's a clean parallel. `LeapSchedule` answers "when does the whole squad stop to eat?" while `SupplySchedule` answers "where do we place feeding stations so the payload never stops?"

The MIP changes from minimizing camp turns (LP on a timeline) to a **set-cover** (which stations tile the march path with food coverage). Here's the full class:

```python
@dataclass  
class SupplyNode:
    """A feeding station along the march route."""
    pieces: list          # pawns assigned here
    hex_idx: int          # station hex
    coverage: set         # path indices this station can feed
    production: float     # net food/turn (after feeding own pawns)
    role: str = SupplyRole.STATION
    group_idx: int = -1   # which pawn group operates this
    tier: int = 0


class SupplySchedule:
    """Optimal supply chain for continuous payload march.
    
    LeapSchedule stops the whole squad to camp.
    SupplySchedule keeps the payload moving — pawn groups
    leapfrog along the route, setting up feeding stations
    whose coverage zones tile the march path.
    
    The MIP is set-cover: select minimum stations so every
    path hex is within cone_range of an active station,
    and assign pawn groups to stations with leapfrog timing.
    """
    
    def __init__(self, payload, pawn_groups, path, nodes, df, payload_diet):
        self.payload = payload          # pieces that march
        self.pawn_groups = pawn_groups  # list of piece-lists
        self.path = path                # TroopPath
        self.nodes = nodes              # active SupplyNodes
        self.df = df                    # per-waypoint coverage DataFrame
        self.payload_diet = payload_diet
        self.timing = {}                # group_idx → list of leg dicts
    
    @property
    def march_turns(self): return len(self.path.hexes)
    
    @property
    def covered_count(self): return int(self.df['covered'].sum())
    
    @property
    def gap_hexes(self): return self.df[~self.df['covered']]
    
    def __repr__(self):
        pct = 100 * self.covered_count / max(self.march_turns, 1)
        return (f"SupplySchedule({self.march_turns} hexes, "
                f"{len(self.nodes)} stations, {pct:.0f}% covered)")
    
    # ── Plan ────────────────────────────────────────────────────────
    @classmethod
    def plan(cls, payload_pieces, pawn_groups, target, grid, elevations,
             food_tiers, countries=None, station_ring=2, cone_range=3):
        """
        Phase 1: Compute payload march path
        Phase 2: Find candidate station hexes near path
        Phase 3: Set-cover MIP with pawn-group assignment
        Phase 4: Leapfrog timing feasibility check
        """
        # ── Phase 1: Path ──
        leader = max(payload_pieces,
                     key=lambda p: RANK_ORDER.get(p.piece_type, 0))
        path = leader.pathfind(target, grid, elevations, countries)
        if not path: return None
        
        W = len(path.hexes)
        payload_diet = sum(p.diet for p in payload_pieces)
        
        # ── Phase 2: Candidates ──
        candidates = cls._candidate_stations(
            path, pawn_groups, grid, elevations, food_tiers,
            countries, station_ring, cone_range)
        if not candidates:
            print("SupplySchedule: no viable stations found"); return None
        
        # ── Phase 3: MIP ──
        G = len(pawn_groups)
        solution = cls._solve_cover(W, candidates, G)
        if solution is None:
            print("SupplySchedule: MIP infeasible"); return None
        
        # ── Build nodes ──
        nodes = []
        for s, c in enumerate(candidates):
            if not solution['active'][s]: continue
            g = solution['group'][s]
            nodes.append(SupplyNode(
                pieces=list(pawn_groups[g]) if g is not None else [],
                hex_idx=c['hex'], coverage=c['coverage'],
                production=c['net_prod'],
                group_idx=g if g is not None else -1,
                tier=c['tier'],
            ))
        
        # ── Coverage DataFrame ──
        rows = []
        for w in range(W):
            covering = [n for n in nodes if w in n.coverage]
            feed = sum(n.production for n in covering)
            rows.append({
                'path_idx': w, 'hex': path.hexes[w],
                'covered': len(covering) > 0,
                'feed_rate': feed,
                'surplus': feed - payload_diet,
                'stations': len(covering),
            })
        df = pd.DataFrame(rows)
        
        sched = cls(payload_pieces, pawn_groups, path, nodes, df, payload_diet)
        
        # ── Phase 4: Timing ──
        sched.timing = sched._compute_timing(grid)
        
        # ── Summary ──
        gaps = len(sched.gap_hexes)
        print(f"SupplySchedule: {W} path hexes, {len(nodes)} stations, "
              f"{sched.covered_count}/{W} covered")
        print(f"  Payload diet: {payload_diet:.1f}/turn"
              + (f", {gaps} uncovered gaps" if gaps else ""))
        for i, n in enumerate(nodes):
            lo, hi = min(n.coverage), max(n.coverage)
            print(f"  Stn {i} (grp {n.group_idx}): ⬡{n.hex_idx} "
                  f"T{n.tier} +{n.production:.1f}/t  "
                  f"path[{lo}..{hi}]")
        for gi, legs in sched.timing.items():
            for leg in legs:
                if not leg['feasible']:
                    print(f"  ⚠️ Grp {gi}: need {leg['transit']}t transit "
                          f"but only {leg['entry']-leg['exit']}t gap "
                          f"(⬡{leg['from_hex']}→⬡{leg['to_hex']})")
        
        return sched
    
    # ── Phase 2: Candidate stations ────────────────────────────────
    @staticmethod
    def _candidate_stations(path, pawn_groups, grid, elevations, food_tiers,
                            countries, station_ring, cone_range):
        """Fertile hexes near the path with their coverage footprints."""
        candidates = []
        seen = set()
        path_set = set(path.hexes)
        
        for whex in path.hexes:
            for ring in range(1, station_ring + 1):
                for hp in HexPosition.origin().ring(ring):
                    sidx = grid.hexposition_to_index(hp, origin_index=whex)
                    if sidx < 0 or sidx in seen: continue
                    if sidx in grid.invalidRegion: continue
                    if elevations[sidx] <= 0: continue
                    if countries is not None and countries[sidx] < 0: continue
                    if sidx in path_set: continue  # keep march lane clear
                    seen.add(sidx)
                    
                    tier = int(food_tiers[sidx]) if 0 <= sidx < len(food_tiers) else 0
                    if tier < 1: continue
                    
                    # Which path indices can this station reach?
                    hp_s = grid.index_to_hexposition(sidx)
                    coverage = set()
                    for pw, phex in enumerate(path.hexes):
                        hp_p = grid.index_to_hexposition(phex)
                        if hp_s.distance(hp_p) <= cone_range:
                            coverage.add(pw)
                    if not coverage: continue
                    
                    # Best net surplus from any pawn group at this tier
                    best_prod = 0
                    for pg in pawn_groups:
                        pg_alive = [p for p in pg
                                    if getattr(p, 'health', 100) > 0]
                        pg_diet = sum(p.diet for p in pg_alive)
                        pg_harvest = sum(p.harvest_strength for p in pg_alive) * tier
                        best_prod = max(best_prod, pg_harvest - pg_diet)
                    if best_prod <= 0: continue
                    
                    candidates.append({
                        'hex': sidx, 'coverage': coverage,
                        'net_prod': best_prod, 'tier': tier,
                    })
        return candidates
    
    # ── Phase 3: Set-cover MIP ─────────────────────────────────────
    @staticmethod
    def _solve_cover(W, candidates, G):
        """Select minimum stations that tile the path, assign to groups."""
        S = len(candidates)
        
        m = pyo.ConcreteModel()
        m.S = pyo.RangeSet(0, S - 1)
        m.G = pyo.RangeSet(0, G - 1)
        
        m.activate = pyo.Var(m.S, domain=pyo.Binary)
        m.assign   = pyo.Var(m.S, m.G, domain=pyo.Binary)
        
        # Every reachable path hex must be covered
        m.cover = pyo.ConstraintList()
        uncoverable = set()
        for w in range(W):
            covers = [s for s in range(S) if w in candidates[s]['coverage']]
            if not covers:
                uncoverable.add(w); continue
            m.cover.add(sum(m.activate[s] for s in covers) >= 1)
        
        # Active station needs exactly one group
        m.needs_group = pyo.Constraint(m.S, rule=lambda m, s:
            m.activate[s] <= sum(m.assign[s, g] for g in m.G))
        m.one_group = pyo.Constraint(m.S, rule=lambda m, s:
            sum(m.assign[s, g] for g in m.G) <= 1)
        
        # Minimize active stations
        m.obj = pyo.Objective(
            expr=sum(m.activate[s] for s in m.S),
            sense=pyo.minimize)
        
        solver = pyo.SolverFactory('appsi_highs')
        result = solver.solve(m)
        if not _solver_ok(result): return None
        
        active = [pyo.value(m.activate[s]) > 0.5 for s in range(S)]
        group = [None] * S
        for s in range(S):
            if active[s]:
                for g in range(G):
                    if pyo.value(m.assign[s, g]) > 0.5:
                        group[s] = g; break
        
        return {'active': active, 'group': group, 'uncoverable': uncoverable}
    
    # ── Phase 4: Leapfrog timing ───────────────────────────────────
    def _compute_timing(self, grid):
        """Check whether each pawn group can physically leapfrog in time.
        
        For group g with stations [s1, s2, ...] ordered by path position:
          exit_turn(s1) + transit(s1→s2) <= entry_turn(s2)
        
        If slack < 0, the payload has to wait (or we need more groups).
        """
        from collections import defaultdict
        by_group = defaultdict(list)
        for n in self.nodes:
            if n.group_idx >= 0:
                by_group[n.group_idx].append(n)
        
        timing = {}
        for gi, stations in by_group.items():
            stations.sort(key=lambda n: min(n.coverage))
            legs = []
            for i in range(len(stations) - 1):
                s1, s2 = stations[i], stations[i + 1]
                exit_turn  = max(s1.coverage)  # last turn s1 is needed
                entry_turn = min(s2.coverage)  # first turn s2 is needed
                hp1 = grid.index_to_hexposition(s1.hex_idx)
                hp2 = grid.index_to_hexposition(s2.hex_idx)
                transit = hp1.distance(hp2)
                slack = entry_turn - exit_turn - transit
                legs.append({
                    'from_hex': s1.hex_idx, 'to_hex': s2.hex_idx,
                    'exit': exit_turn, 'entry': entry_turn,
                    'transit': transit, 'slack': slack,
                    'feasible': slack >= 0,
                })
            timing[gi] = legs
        return timing
```

The structural parallel to `LeapSchedule`:

| | **LeapSchedule** | **SupplySchedule** |
|---|---|---|
| Unit | whole squad | payload + N pawn groups |
| Decision | camp turns at waypoints | station positions along path |
| MIP type | LP (minimize camp turns) | set-cover (tile path with coverage) |
| Constraint | food ≥ min at each waypoint | every path hex covered by a station |
| Refinement | `optimal_camp_layout` at camp stops | could run it at each station too |
| Output | `(waypoint, camp_turns)` schedule | `(station_hex, group_idx)` placement |

The timing check is the key addition — it catches the case where a pawn group can't physically leapfrog fast enough. When `slack < 0`, you either need a third group (to cover the gap while the other transits) or the payload has to pause.

Next natural step: when timing is infeasible, either bump the group count or feed the timing constraint back into the MIP as an ordering constraint. Want to try it on your 20×20 map first to see what the coverage looks like?

In [ ]:
@dataclass  
class SupplyNode:
    """A feeding station along the march route."""
    pieces: list          # pawns assigned here
    hex_idx: int          # station hex
    coverage: set         # path indices this station can feed
    production: float     # net food/turn (after feeding own pawns)
    role: str = SupplyRole.STATION
    group_idx: int = -1   # which pawn group operates this
    tier: int = 0


class SupplySchedule:
    """Optimal supply chain for continuous payload march.
    
    LeapSchedule stops the whole squad to camp.
    SupplySchedule keeps the payload moving — pawn groups
    leapfrog along the route, setting up feeding stations
    whose coverage zones tile the march path.
    
    The MIP is set-cover: select minimum stations so every
    path hex is within cone_range of an active station,
    and assign pawn groups to stations with leapfrog timing.
    """
    
    def __init__(self, payload, pawn_groups, path, nodes, df, payload_diet):
        self.payload = payload          # pieces that march
        self.pawn_groups = pawn_groups  # list of piece-lists
        self.path = path                # TroopPath
        self.nodes = nodes              # active SupplyNodes
        self.df = df                    # per-waypoint coverage DataFrame
        self.payload_diet = payload_diet
        self.timing = {}                # group_idx → list of leg dicts
    
    @property
    def march_turns(self): return len(self.path.hexes)
    
    @property
    def covered_count(self): return int(self.df['covered'].sum())
    
    @property
    def gap_hexes(self): return self.df[~self.df['covered']]
    
    def __repr__(self):
        pct = 100 * self.covered_count / max(self.march_turns, 1)
        return (f"SupplySchedule({self.march_turns} hexes, "
                f"{len(self.nodes)} stations, {pct:.0f}% covered)")
    
    # ── Plan ────────────────────────────────────────────────────────
    @classmethod
    def plan(cls, payload_pieces, pawn_groups, target, grid, elevations,
             food_tiers, countries=None, station_ring=2, cone_range=3):
        """
        Phase 1: Compute payload march path
        Phase 2: Find candidate station hexes near path
        Phase 3: Set-cover MIP with pawn-group assignment
        Phase 4: Leapfrog timing feasibility check
        """
        # ── Phase 1: Path ──
        leader = max(payload_pieces,
                     key=lambda p: RANK_ORDER.get(p.piece_type, 0))
        path = leader.pathfind(target, grid, elevations, countries)
        if not path: return None
        
        W = len(path.hexes)
        payload_diet = sum(p.diet for p in payload_pieces)
        
        # ── Phase 2: Candidates ──
        candidates = cls._candidate_stations(
            path, pawn_groups, grid, elevations, food_tiers,
            countries, station_ring, cone_range)
        if not candidates:
            print("SupplySchedule: no viable stations found"); return None
        
        # ── Phase 3: MIP ──
        G = len(pawn_groups)
        solution = cls._solve_cover(W, candidates, G)
        if solution is None:
            print("SupplySchedule: MIP infeasible"); return None
        
        # ── Build nodes ──
        nodes = []
        for s, c in enumerate(candidates):
            if not solution['active'][s]: continue
            g = solution['group'][s]
            nodes.append(SupplyNode(
                pieces=list(pawn_groups[g]) if g is not None else [],
                hex_idx=c['hex'], coverage=c['coverage'],
                production=c['net_prod'],
                group_idx=g if g is not None else -1,
                tier=c['tier'],
            ))
        
        # ── Coverage DataFrame ──
        rows = []
        for w in range(W):
            covering = [n for n in nodes if w in n.coverage]
            feed = sum(n.production for n in covering)
            rows.append({
                'path_idx': w, 'hex': path.hexes[w],
                'covered': len(covering) > 0,
                'feed_rate': feed,
                'surplus': feed - payload_diet,
                'stations': len(covering),
            })
        df = pd.DataFrame(rows)
        
        sched = cls(payload_pieces, pawn_groups, path, nodes, df, payload_diet)
        
        # ── Phase 4: Timing ──
        sched.timing = sched._compute_timing(grid)
        
        # ── Summary ──
        gaps = len(sched.gap_hexes)
        print(f"SupplySchedule: {W} path hexes, {len(nodes)} stations, "
              f"{sched.covered_count}/{W} covered")
        print(f"  Payload diet: {payload_diet:.1f}/turn"
              + (f", {gaps} uncovered gaps" if gaps else ""))
        for i, n in enumerate(nodes):
            lo, hi = min(n.coverage), max(n.coverage)
            print(f"  Stn {i} (grp {n.group_idx}): ⬡{n.hex_idx} "
                  f"T{n.tier} +{n.production:.1f}/t  "
                  f"path[{lo}..{hi}]")
        for gi, legs in sched.timing.items():
            for leg in legs:
                if not leg['feasible']:
                    print(f"  ⚠️ Grp {gi}: need {leg['transit']}t transit "
                          f"but only {leg['entry']-leg['exit']}t gap "
                          f"(⬡{leg['from_hex']}→⬡{leg['to_hex']})")
        
        return sched
    
    # ── Phase 2: Candidate stations ────────────────────────────────
    @staticmethod
    def _candidate_stations(path, pawn_groups, grid, elevations, food_tiers,
                            countries, station_ring, cone_range):
        """Fertile hexes near the path with their coverage footprints."""
        candidates = []
        seen = set()
        path_set = set(path.hexes)
        
        for whex in path.hexes:
            for ring in range(1, station_ring + 1):
                for hp in HexPosition.origin().ring(ring):
                    sidx = grid.hexposition_to_index(hp, origin_index=whex)
                    if sidx < 0 or sidx in seen: continue
                    if sidx in grid.invalidRegion: continue
                    if elevations[sidx] <= 0: continue
                    if countries is not None and countries[sidx] < 0: continue
                    if sidx in path_set: continue  # keep march lane clear
                    seen.add(sidx)
                    
                    tier = int(food_tiers[sidx]) if 0 <= sidx < len(food_tiers) else 0
                    if tier < 1: continue
                    
                    # Which path indices can this station reach?
                    hp_s = grid.index_to_hexposition(sidx)
                    coverage = set()
                    for pw, phex in enumerate(path.hexes):
                        hp_p = grid.index_to_hexposition(phex)
                        if hp_s.distance(hp_p) <= cone_range:
                            coverage.add(pw)
                    if not coverage: continue
                    
                    # Best net surplus from any pawn group at this tier
                    best_prod = 0
                    for pg in pawn_groups:
                        pg_alive = [p for p in pg
                                    if getattr(p, 'health', 100) > 0]
                        pg_diet = sum(p.diet for p in pg_alive)
                        pg_harvest = sum(p.harvest_strength for p in pg_alive) * tier
                        best_prod = max(best_prod, pg_harvest - pg_diet)
                    if best_prod <= 0: continue
                    
                    candidates.append({
                        'hex': sidx, 'coverage': coverage,
                        'net_prod': best_prod, 'tier': tier,
                    })
        return candidates
    
    # ── Phase 3: Set-cover MIP ─────────────────────────────────────
    @staticmethod
    def _solve_cover(W, candidates, G):
        """Select minimum stations that tile the path, assign to groups."""
        S = len(candidates)
        
        m = pyo.ConcreteModel()
        m.S = pyo.RangeSet(0, S - 1)
        m.G = pyo.RangeSet(0, G - 1)
        
        m.activate = pyo.Var(m.S, domain=pyo.Binary)
        m.assign   = pyo.Var(m.S, m.G, domain=pyo.Binary)
        
        # Every reachable path hex must be covered
        m.cover = pyo.ConstraintList()
        uncoverable = set()
        for w in range(W):
            covers = [s for s in range(S) if w in candidates[s]['coverage']]
            if not covers:
                uncoverable.add(w); continue
            m.cover.add(sum(m.activate[s] for s in covers) >= 1)
        
        # Active station needs exactly one group
        m.needs_group = pyo.Constraint(m.S, rule=lambda m, s:
            m.activate[s] <= sum(m.assign[s, g] for g in m.G))
        m.one_group = pyo.Constraint(m.S, rule=lambda m, s:
            sum(m.assign[s, g] for g in m.G) <= 1)
        
        # Minimize active stations
        m.obj = pyo.Objective(
            expr=sum(m.activate[s] for s in m.S),
            sense=pyo.minimize)
        
        solver = pyo.SolverFactory('appsi_highs')
        result = solver.solve(m)
        if not _solver_ok(result): return None
        
        active = [pyo.value(m.activate[s]) > 0.5 for s in range(S)]
        group = [None] * S
        for s in range(S):
            if active[s]:
                for g in range(G):
                    if pyo.value(m.assign[s, g]) > 0.5:
                        group[s] = g; break
        
        return {'active': active, 'group': group, 'uncoverable': uncoverable}
    
    # ── Phase 4: Leapfrog timing ───────────────────────────────────
    def _compute_timing(self, grid):
        """Check whether each pawn group can physically leapfrog in time.
        
        For group g with stations [s1, s2, ...] ordered by path position:
          exit_turn(s1) + transit(s1→s2) <= entry_turn(s2)
        
        If slack < 0, the payload has to wait (or we need more groups).
        """
        from collections import defaultdict
        by_group = defaultdict(list)
        for n in self.nodes:
            if n.group_idx >= 0:
                by_group[n.group_idx].append(n)
        
        timing = {}
        for gi, stations in by_group.items():
            stations.sort(key=lambda n: min(n.coverage))
            legs = []
            for i in range(len(stations) - 1):
                s1, s2 = stations[i], stations[i + 1]
                exit_turn  = max(s1.coverage)  # last turn s1 is needed
                entry_turn = min(s2.coverage)  # first turn s2 is needed
                hp1 = grid.index_to_hexposition(s1.hex_idx)
                hp2 = grid.index_to_hexposition(s2.hex_idx)
                transit = hp1.distance(hp2)
                slack = entry_turn - exit_turn - transit
                legs.append({
                    'from_hex': s1.hex_idx, 'to_hex': s2.hex_idx,
                    'exit': exit_turn, 'entry': entry_turn,
                    'transit': transit, 'slack': slack,
                    'feasible': slack >= 0,
                })
            timing[gi] = legs
        return timing


How do we test this? How do we visualize it?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Two pieces: a test harness to split the squad and run the planner, then an overlay to draw the result.

**Test first** — split the queen guard into payload + 2 pawn groups and pick a target:

```python
# Split squad into payload + pawn groups
alive = myStuff.queenGuard.alive
payload = [p for p in alive if p.piece_type in (PieceType.QUEEN, PieceType.BISHOP)]
pawns = [p for p in alive if p.piece_type == PieceType.PAWN]

# Split pawns into 2 groups (roughly equal)
mid = len(pawns) // 2
pawn_groups = [pawns[:mid], pawns[mid:]]

print(f"Payload: {[p.name for p in payload]} (diet {sum(p.diet for p in payload):.1f})")
for i, pg in enumerate(pawn_groups):
    harv = sum(p.harvest_strength for p in pg)
    diet = sum(p.diet for p in pg)
    print(f"Group {i}: {[p.name for p in pg]} (harvest {harv:.1f}, diet {diet:.1f})")

# Pick a target hex — somewhere across the map on land
elevs = myStuff.terr.elevations
target = None
for idx in range(len(elevs) - 1, -1, -1):
    if elevs[idx] > 0 and idx not in myStuff.grid.invalidRegion:
        target = idx
        break
print(f"Target: ⬡{target}")
```

Run that first to make sure the split looks right, then:

```python
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

sched = SupplySchedule.plan(
    payload_pieces=payload,
    pawn_groups=pawn_groups,
    target=target,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    countries=myStuff.terr.fields.get('country'),
    station_ring=2,
    cone_range=3,
)
print(sched)
print(sched.df)
```

**Then the overlay** — this draws path, stations, coverage zones, and feed arcs:

```python
def SupplyScheduleOverlay(sched: SupplySchedule) -> OverlaySpec:
    """Visualize a supply schedule on the hex map.
    
    - Path hexes: green border (covered) or red border (gap)
    - Station hexes: gold fill with tier label
    - Coverage zones: faint colored polygons per group
    - Feed arcs: dashed lines from station to path
    """
    GROUP_COLORS = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12']
    
    def render(ctx):
        grid = ctx.grid
        parts = []
        
        # ── Path hexes ──
        for _, row in sched.df.iterrows():
            h = grid.hexes[int(row['hex'])]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            color = '#2ecc71' if row['covered'] else '#e74c3c'
            parts.append(
                f'<polygon points="{pts}" fill="none" '
                f'stroke="{color}" stroke-width="2" opacity="0.8"/>')
        
        # ── Coverage zones (faint fill per group) ──
        for node in sched.nodes:
            gc = GROUP_COLORS[node.group_idx % len(GROUP_COLORS)]
            for pw in node.coverage:
                if pw >= len(sched.path.hexes): continue
                phex = sched.path.hexes[pw]
                h = grid.hexes[phex]
                pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
                parts.append(
                    f'<polygon points="{pts}" fill="{gc}" opacity="0.15"/>')
        
        # ── Station hexes ──
        for node in sched.nodes:
            h = grid.hexes[node.hex_idx]
            cx, cy = h.center.x, h.center.y
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            gc = GROUP_COLORS[node.group_idx % len(GROUP_COLORS)]
            parts.append(
                f'<polygon points="{pts}" fill="{gc}" opacity="0.5" '
                f'stroke="{gc}" stroke-width="3"/>')
            parts.append(
                f'<text x="{cx:.0f}" y="{cy:.0f}" text-anchor="middle" '
                f'dominant-baseline="central" font-size="10" '
                f'font-weight="bold" fill="white">T{node.tier}</text>')
        
        # ── Feed arcs: dashed line from station to nearest path hex ──
        for node in sched.nodes:
            sh = grid.hexes[node.hex_idx]
            sx, sy = sh.center.x, sh.center.y
            mid_pw = sorted(node.coverage)[len(node.coverage) // 2]
            if mid_pw < len(sched.path.hexes):
                ph = grid.hexes[sched.path.hexes[mid_pw]]
                px, py = ph.center.x, ph.center.y
                gc = GROUP_COLORS[node.group_idx % len(GROUP_COLORS)]
                parts.append(
                    f'<line x1="{sx:.0f}" y1="{sy:.0f}" '
                    f'x2="{px:.0f}" y2="{py:.0f}" '
                    f'stroke="{gc}" stroke-width="2" '
                    f'stroke-dasharray="4,3" opacity="0.6"/>')
        
        # ── Path direction markers (small arrows every 5 hexes) ──
        for i in range(0, len(sched.path.hexes) - 1, 5):
            h1 = grid.hexes[sched.path.hexes[i]]
            h2 = grid.hexes[sched.path.hexes[min(i + 1, len(sched.path.hexes) - 1)]]
            x1, y1 = h1.center.x, h1.center.y
            x2, y2 = h2.center.x, h2.center.y
            parts.append(
                f'<circle cx="{x1:.0f}" cy="{y1:.0f}" r="3" '
                f'fill="#333" opacity="0.5"/>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("supply_schedule", render, priority=55)
```

Then display it:

```python
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    RiverOverlay(max_width=5),
    SupplyScheduleOverlay(sched),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
)
```

The overlay shows:
- **Green borders** on path hexes that have food coverage, **red** for gaps
- **Colored fills** per pawn group (blue/red) showing which group covers which stretch
- **Gold station hexes** with tier labels
- **Dashed arcs** from each station to its midpoint on the path

Start with the first code block (the squad split) and see if the numbers make sense before running the full planner.

## pawn cover subgame.
I think we need to build small pieces out. The first is what I will call a pawn covered route (pcr). a route is a sequence of conected hexes. we want to have each of the these hexes covered by two pawns such that the amount the pawns give could feed a queen. a catch is that the pawns can't be on the route. lets intitiall just have the magic hand to place the pawns, but once place we will simulate if the queen can traverse. This is a bit of a wherehouse location problem. but I think we need to simulate solutions for testing.

I know I am pivioting to this sub problem, but I want to see about the queen traversing between settlements

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good pivot — this is the right atomic building block. Get this working and everything else composes on top of it.

Let's build it in three pieces:

**Piece 1: Route + pawn placement (the "magic hand")**

```python
def pawn_covered_route(start_hex, end_hex, grid, elevations, food_tiers,
                       pawn_harvest=2.0, pawn_diet=2.0, queen_diet=7.0,
                       cone_range=2, countries=None):
    """Find a route and place pawns to feed a queen traversing it.
    
    Returns dict with route, pawn placements, coverage info, or None.
    """
    # ── Route: pathfind start → end ──
    # Use HexPosition BFS on land hexes
    from collections import deque
    
    parent = {start_hex: None}
    queue = deque([start_hex])
    while queue:
        cur = queue.popleft()
        if cur == end_hex: break
        for nb in grid.neighborsOf(cur):
            if nb in parent: continue
            if nb < 0 or nb >= len(elevations): continue
            if nb in grid.invalidRegion: continue
            if elevations[nb] <= 0: continue
            if countries is not None and countries[nb] < 0: continue
            parent[nb] = cur
            queue.append(nb)
    
    if end_hex not in parent:
        print(f"No land route ⬡{start_hex} → ⬡{end_hex}"); return None
    
    # Reconstruct path
    route = []
    cur = end_hex
    while cur is not None:
        route.append(cur)
        cur = parent[cur]
    route.reverse()
    route_set = set(route)
    W = len(route)
    
    # ── Candidate pawn positions: adjacent to route, not ON route, fertile ──
    candidates = []
    seen = set()
    for whex in route:
        for nb in grid.neighborsOf(whex):
            if nb in seen or nb in route_set: continue
            if nb < 0 or nb >= len(elevations): continue
            if nb in grid.invalidRegion: continue
            if elevations[nb] <= 0: continue
            if countries is not None and countries[nb] < 0: continue
            seen.add(nb)
            
            tier = int(food_tiers[nb]) if 0 <= nb < len(food_tiers) else 0
            if tier < 1: continue
            
            # Coverage: which route indices can this pawn reach?
            hp_nb = grid.index_to_hexposition(nb)
            coverage = set()
            for wi, whex2 in enumerate(route):
                hp_w = grid.index_to_hexposition(whex2)
                if hp_nb.distance(hp_w) <= cone_range:
                    coverage.add(wi)
            if not coverage: continue
            
            net_export = pawn_harvest * tier - pawn_diet  # surplus per turn
            candidates.append({
                'hex': nb, 'tier': tier, 'coverage': coverage,
                'export': max(0, net_export),
            })
    
    if not candidates:
        print("No fertile candidates adjacent to route"); return None
    
    # ── Set-cover MIP: minimum pawns to cover every route hex ──
    S = len(candidates)
    m = pyo.ConcreteModel()
    m.S = pyo.RangeSet(0, S - 1)
    m.activate = pyo.Var(m.S, domain=pyo.Binary)
    
    # Every route hex must be covered by at least one pawn
    # whose export can contribute to queen's diet
    m.cover = pyo.ConstraintList()
    uncoverable = set()
    for w in range(W):
        covers = [s for s in range(S)
                  if w in candidates[s]['coverage'] and candidates[s]['export'] > 0]
        if not covers:
            uncoverable.add(w); continue
        # Enough export to feed the queen at this step
        m.cover.add(
            sum(candidates[s]['export'] * m.activate[s] for s in covers) >= queen_diet)
    
    m.obj = pyo.Objective(
        expr=sum(m.activate[s] for s in m.S), sense=pyo.minimize)
    
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    if not _solver_ok(result):
        print(f"MIP infeasible — {len(uncoverable)} uncoverable hexes"); return None
    
    # ── Extract placements ──
    placements = []
    for s in range(S):
        if pyo.value(m.activate[s]) > 0.5:
            placements.append(candidates[s])
    
    # ── Coverage check ──
    covered = set()
    for p in placements:
        covered |= p['coverage']
    
    gaps = [w for w in range(W) if w not in covered]
    
    print(f"PCR: route {W} hexes, {len(placements)} pawns, "
          f"{len(covered)}/{W} covered"
          + (f", {len(gaps)} gaps" if gaps else " ✅"))
    for p in placements:
        lo, hi = min(p['coverage']), max(p['coverage'])
        print(f"  ⬡{p['hex']} T{p['tier']} export={p['export']:.1f}/t  path[{lo}..{hi}]")
    
    return {
        'route': route,
        'placements': placements,
        'covered': covered,
        'gaps': gaps,
        'uncoverable': uncoverable,
        'queen_diet': queen_diet,
    }
```

**Piece 2: Overlay to see it**

```python
def PCROverlay(pcr: dict) -> OverlaySpec:
    """Visualize a pawn-covered route."""
    def render(ctx):
        grid = ctx.grid
        parts = []
        route = pcr['route']
        covered = pcr['covered']
        
        # Route hexes: green if covered, red if gap
        for w, hidx in enumerate(route):
            h = grid.hexes[hidx]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in h.vertices())
            color = '#2ecc71' if w in covered else '#e74c3c'
            parts.append(
                f'<polygon points="{pts}" fill="{color}" opacity="0.25" '
                f'stroke="{color}" stroke-width="2.5"/>')
        
        # Route direction: line through centers
        for i in range(len(route) - 1):
            h1, h2 = grid.hexes[route[i]], grid.hexes[route[i+1]]
            parts.append(
                f'<line x1="{h1.center.x:.0f}" y1="{h1.center.y:.0f}" '
                f'x2="{h2.center.x:.0f}" y2="{h2.center.y:.0f}" '
                f'stroke="#333" stroke-width="1.5" opacity="0.4"/>')
        
        # Pawn hexes: blue fill with tier label
        for p in pcr['placements']:
            h = grid.hexes[p['hex']]
            cx, cy = h.center.x, h.center.y
            pts = " ".join(f"{v.x:.0f},{v.y:.0f}" for v in h.vertices())
            parts.append(
                f'<polygon points="{pts}" fill="#3498db" opacity="0.45" '
                f'stroke="#2980b9" stroke-width="2"/>')
            parts.append(
                f'<text x="{cx:.0f}" y="{cy:.0f}" text-anchor="middle" '
                f'dominant-baseline="central" font-size="10" '
                f'font-weight="bold" fill="white">♟T{p["tier"]}</text>')
            
            # Dashed arc to middle of coverage
            mid_w = sorted(p['coverage'])[len(p['coverage']) // 2]
            if mid_w < len(route):
                ph = grid.hexes[route[mid_w]]
                parts.append(
                    f'<line x1="{cx:.0f}" y1="{cy:.0f}" '
                    f'x2="{ph.center.x:.0f}" y2="{ph.center.y:.0f}" '
                    f'stroke="#2980b9" stroke-width="1.5" '
                    f'stroke-dasharray="4,3" opacity="0.5"/>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("pcr", render, priority=55)
```

**Piece 3: Test it**

```python
# Get the two settlement locations
s0 = myStuff.board.kingdoms[0].settlements[0].location
s1 = myStuff.board.kingdoms[1].settlements[0].location
print(f"Route: ⬡{s0} → ⬡{s1}")

fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

pcr = pawn_covered_route(
    s0, s1, myStuff.grid, myStuff.terr.elevations, fy.tiers,
    queen_diet=7.0, cone_range=2,
)
```

Then visualize:

```python
TerrainDisplay(
    TerrainOverlay(),
    SettlementOverlay(),
    RiverOverlay(max_width=5),
    PCROverlay(pcr),
    FoodOverlay(),
    terrain=myStuff.terr,
    board=myStuff.board,
    basins=myStuff.basin,
    debug=not showDemo
)
```

Start with Piece 1 + the test — if the MIP finds a solution and the numbers look right, add the overlay. If it says "infeasible", we'll know the route passes through barren terrain and need to decide: reroute through fertile land, or accept gaps and use bishop buffer capacity to bridge them.

In [ ]:
def pawn_covered_route(start_hex, end_hex, grid, elevations, food_tiers,
                       pawn_harvest=2.0, pawn_diet=2.0, queen_diet=7.0,
                       cone_range=2, countries=None):
    """Find a route and place pawns to feed a queen traversing it.
    
    Returns dict with route, pawn placements, coverage info, or None.
    """
    from collections import deque
    
    parent = {start_hex: None}
    queue = deque([start_hex])
    while queue:
        cur = queue.popleft()
        if cur == end_hex: break
        for nb in grid.neighborsOf(cur):
            if nb in parent: continue
            if nb < 0 or nb >= len(elevations): continue
            if nb in grid.invalidRegion: continue
            if elevations[nb] <= 0: continue
            if countries is not None and countries[nb] < 0: continue
            parent[nb] = cur
            queue.append(nb)
    
    if end_hex not in parent:
        print(f"No land route ⬡{start_hex} → ⬡{end_hex}"); return None
    
    # Reconstruct path
    route = []
    cur = end_hex
    while cur is not None:
        route.append(cur)
        cur = parent[cur]
    route.reverse()
    route_set = set(route)
    W = len(route)
    
    # ── Candidate pawn positions: adjacent to route, not ON route, fertile ──
    candidates = []
    seen = set()
    for whex in route:
        for nb in grid.neighborsOf(whex):
            if nb in seen or nb in route_set: continue
            if nb < 0 or nb >= len(elevations): continue
            if nb in grid.invalidRegion: continue
            if elevations[nb] <= 0: continue
            if countries is not None and countries[nb] < 0: continue
            seen.add(nb)
            
            tier = int(food_tiers[nb]) if 0 <= nb < len(food_tiers) else 0
            if tier < 1: continue
            
            # Coverage: which route indices can this pawn reach?
            hp_nb = grid.index_to_hexposition(nb)
            coverage = set()
            for wi, whex2 in enumerate(route):
                hp_w = grid.index_to_hexposition(whex2)
                if hp_nb.distance(hp_w) <= cone_range:
                    coverage.add(wi)
            if not coverage: continue
            
            net_export = pawn_harvest * tier - pawn_diet  # surplus per turn
            candidates.append({
                'hex': nb, 'tier': tier, 'coverage': coverage,
                'export': max(0, net_export),
            })
    
    if not candidates:
        print("No fertile candidates adjacent to route"); return None
    
    # ── Set-cover MIP: minimum pawns to cover every route hex ──
    S = len(candidates)
    m = pyo.ConcreteModel()
    m.S = pyo.RangeSet(0, S - 1)
    m.placed = pyo.Var(m.S, domain=pyo.Binary)
    
    m.cover = pyo.ConstraintList()
    uncoverable = set()
    for w in range(W):
        covers = [s for s in range(S)
                  if w in candidates[s]['coverage'] and candidates[s]['export'] > 0]
        if not covers:
            uncoverable.add(w); continue
        m.cover.add(
            sum(candidates[s]['export'] * m.placed[s] for s in covers) >= queen_diet)
    
    m.obj = pyo.Objective(
        expr=sum(m.placed[s] for s in m.S), sense=pyo.minimize)
    
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m)
    if not _solver_ok(result):
        print(f"MIP infeasible — {len(uncoverable)} uncoverable hexes"); return None
    
    # ── Extract placements ──
    placements = []
    for s in range(S):
        if pyo.value(m.placed[s]) > 0.5:
            placements.append(candidates[s])
    
    # ── Coverage check ──
    covered = set()
    for p in placements:
        covered |= p['coverage']
    
    gaps = [w for w in range(W) if w not in covered]
    
    print(f"PCR: route {W} hexes, {len(placements)} pawns, "
          f"{len(covered)}/{W} covered"
          + (f", {len(gaps)} gaps" if gaps else " ✅"))
    for p in placements:
        lo, hi = min(p['coverage']), max(p['coverage'])
        print(f"  ⬡{p['hex']} T{p['tier']} export={p['export']:.1f}/t  path[{lo}..{hi}]")
    
    return {
        'route': route,
        'placements': placements,
        'covered': covered,
        'gaps': gaps,
        'uncoverable': uncoverable,
        'queen_diet': queen_diet,
    }


In [ ]:
rebootWorld()
myStuff.apply_food_profile(soil_scale=7)
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

In [ ]:
cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)
print(cities)

In [ ]:
start = cities[0].location
finish = cities[1].location
pawn_covered_route(start,finish,grid=myStuff.grid,elevations=myStuff.terr.elevations,food_tiers=fy.tiers)

In [ ]:
# ── Apply Profile ──────────────────────────────────────────────
FOOD_PROFILE_V1 = {
    'queen_diet':     6.4,
    'queen_capacity': 24.0,
    'queen_harvest':  0.2,   # queens barely forage
    'pawn_diet':      1.3,
    'pawn_harvest':   0.57,
    'pawn_foresight': 4,
    'pawn_capacity':  14.0,
    # Bishop: TBD (optimize for buffer turns later)
    'bishop_diet':    3.5,
    'bishop_capacity': 140.0,
    'bishop_harvest': 0.7,
}

def apply_food_profile(game, profile=None):
    """Apply food profile to all pieces in the game."""
    p = profile or FOOD_PROFILE_V1
    for piece in game.pieces:
        t = piece.piece_type.name.lower()
        for attr in ['diet', 'capacity', 'harvest', 'foresight']:
            key = f'{t}_{attr}'
            if key in p:
                setattr(piece, attr, p[key])
                if attr == 'capacity':
                    piece.food = min(piece.food, p[key])  # clamp current food


So I created some new profiles which requires the pawn to have a sight of 4. can you update the demo and see if it can find a feasible solution.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The infeasibility was almost certainly the narrow `cone_range=2` — with 4 the coverage footprint per pawn is massively bigger. Here's the updated test:

```python
p = FOOD_PROFILE_V1

# Quick check the math
for tier in range(1, 8):
    export = p['pawn_harvest'] * tier - p['pawn_diet']
    pawns_needed = math.ceil(p['queen_diet'] / max(export, 0.01)) if export > 0 else -1
    print(f"  Tier {tier}: export={export:+.2f}/t → {pawns_needed} pawns for queen")
```

Run that first to sanity-check, then:

```python
rebootWorld()

fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()
start = cities[0].location
finish = cities[1].location

pcr = pawn_covered_route(
    start, finish,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pawn_harvest=p['pawn_harvest'],
    pawn_diet=p['pawn_diet'],
    queen_diet=p['queen_diet'],
    cone_range=4,  # pawn sight from new profile
)
```

Also fix the crash — the solver blows up before we can check feasibility. Patch the MIP solve in `pawn_covered_route` to use `load_solutions=False`:

```python
    solver = pyo.SolverFactory('appsi_highs')
    result = solver.solve(m, load_solutions=False)
    if not _solver_ok(result):
        print(f"MIP infeasible — {len(uncoverable)} uncoverable hexes")
        return None
    m.solutions.load_from(result)
```

That three-line pattern (solve with `load_solutions=False`, check, then load) is the safe version — it's the same pattern `LeapSchedule._solve_schedule` already uses. Worth updating the function before re-running.

In [ ]:
p = FOOD_PROFILE_V1

# Quick check the math
for tier in range(1, 8):
    export = p['pawn_harvest'] * tier - p['pawn_diet']
    pawns_needed = math.ceil(p['queen_diet'] / max(export, 0.01)) if export > 0 else -1
    print(f"  Tier {tier}: export={export:+.2f}/t → {pawns_needed} pawns for queen")


Is that ok? the goal was 2.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Not quite — at tier 7 (the best land) it still needs 3 pawns. For a "2 pawns sustain the queen" target, we need each pawn to export ≥ `queen_diet / 2 = 3.2/turn`.

The equation: `pawn_harvest × tier - pawn_diet ≥ 3.2`

The question is: at which tier should 2 pawns work?

| Target tier | Required `pawn_harvest` | (with `pawn_diet=1.3`) |
|---|---|---|
| Tier 7 (best) | `(3.2 + 1.3) / 7` = **0.643** | easy, only top land |
| Tier 5 (good) | `(3.2 + 1.3) / 5` = **0.90** | decent common land |
| Tier 4 (mid) | `(3.2 + 1.3) / 4` = **1.125** | works almost everywhere |

Bumping `pawn_harvest` from `0.57` → `0.65` gets you 2 pawns at tier 7 only. Bumping to `0.9` gets 2 pawns on any tier-5+ hex. 

Another lever — `queen_diet=6.4` is pretty hungry. Dropping to `5.0` with the current `0.57` harvest:

```
Tier 5: export=1.55 → ceil(5.0/1.55) = 4 pawns
Tier 7: export=2.69 → ceil(5.0/2.69) = 2 pawns ✅
```

So the two knobs are:
- **`pawn_harvest`**: how good are pawns at farming?
- **`queen_diet`**: how expensive is the queen?

What's the design intent — should 2 pawns work on *good* land (tier 5+), or only the *best* land (tier 7)?

So can we use either pyomo or optima so that we can adjust the harvest of pawns and possibly other factors so the queen can make it between cities. I think giving them a range of 4 means that they wouldn't have to pick low tier hexes, but clearly there needs to be some combination of higher tiers that makes sense with even a tier 7 to probably support a queen all by itself.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great idea — let Optuna find the profile that makes the route work. We'll define soft constraints as penalty terms so it can explore the tradeoff space:

```python
def pcr_objective(trial):
    """Optuna objective: find food profile where PCR is feasible with few pawns."""
    
    # ── Search space ──
    pawn_harvest = trial.suggest_float('pawn_harvest', 0.5, 2.0)
    pawn_diet    = trial.suggest_float('pawn_diet', 0.8, 2.0)
    queen_diet   = trial.suggest_float('queen_diet', 4.0, 8.0)
    cone_range   = 4  # fixed: pawn sight = 4
    
    # ── Hard design constraints (as penalties) ──
    penalty = 0
    
    # 1) Single pawn on tier-7 should sustain the queen
    solo_t7_export = pawn_harvest * 7 - pawn_diet
    if solo_t7_export < queen_diet:
        penalty += 50 * (queen_diet - solo_t7_export)
    
    # 2) Two pawns on tier-5 should sustain the queen
    duo_t5_export = 2 * (pawn_harvest * 5 - pawn_diet)
    if duo_t5_export < queen_diet:
        penalty += 30 * (queen_diet - duo_t5_export)
    
    # 3) Pawns should self-sustain on tier-3+ (not starve while feeding)
    self_sustain_t3 = pawn_harvest * 3 - pawn_diet
    if self_sustain_t3 < 0:
        penalty += 20 * abs(self_sustain_t3)
    
    # 4) Queen should feel expensive (diet ratio queen/pawn ≥ 3)
    diet_ratio = queen_diet / max(pawn_diet, 0.1)
    if diet_ratio < 3.0:
        penalty += 10 * (3.0 - diet_ratio)
    
    if penalty > 0:
        return 1000 + penalty  # infeasible design, but guide the search
    
    # ── Run the actual PCR ──
    try:
        pcr = pawn_covered_route(
            start, finish,
            grid=myStuff.grid,
            elevations=myStuff.terr.elevations,
            food_tiers=fy.tiers,
            pawn_harvest=pawn_harvest,
            pawn_diet=pawn_diet,
            queen_diet=queen_diet,
            cone_range=cone_range,
        )
    except Exception as e:
        return 999  # solver crash
    
    if pcr is None:
        return 998  # infeasible route
    
    n_pawns = len(pcr['placements'])
    n_gaps = len(pcr['gaps'])
    
    # Objective: minimize pawns, heavily penalize gaps
    return n_pawns + 100 * n_gaps
```

Then run the study:

```python
# Suppress solver spam during search
import logging as _log
_log.getLogger('pyomo').setLevel(_log.ERROR)

study = optuna.create_study(direction='minimize', study_name='food_profile')
study.optimize(pcr_objective, n_trials=80, show_progress_bar=True)

# ── Results ──
best = study.best_trial
print(f"\nBest: {best.value:.0f} pawns")
for k, v in best.params.items():
    print(f"  {k}: {v:.3f}")

# Check the design constraints
p = best.params
t7 = p['pawn_harvest'] * 7 - p['pawn_diet']
t5 = p['pawn_harvest'] * 5 - p['pawn_diet']
print(f"\n  Solo pawn @T7 exports: {t7:.2f} (queen needs {p['queen_diet']:.2f})"
      f" {'✅' if t7 >= p['queen_diet'] else '❌'}")
print(f"  Two pawns @T5 export: {2*t5:.2f} (queen needs {p['queen_diet']:.2f})"
      f" {'✅' if 2*t5 >= p['queen_diet'] else '❌'}")
print(f"  Pawn self-sustains @T3: {p['pawn_harvest']*3 - p['pawn_diet']:.2f}"
      f" {'✅' if p['pawn_harvest']*3 > p['pawn_diet'] else '❌'}")
print(f"  Diet ratio queen/pawn: {p['queen_diet']/p['pawn_diet']:.1f}×")
```

The design constraints baked in are:

| Rule | Why |
|---|---|
| 1 pawn @ tier 7 sustains queen | Top land = strategic chokepoint |
| 2 pawns @ tier 5 sustain queen | Good land is playable with small escort |
| Pawn self-sustains @ tier 3 | Pawns don't starve on mediocre terrain |
| Queen/pawn diet ratio ≥ 3× | Queen feels expensive, pawns feel cheap |

Once Optuna finds good params, rerun the PCR and visualize it. You may want to add more constraints (like bishop buffer turns, or max total pawns for a given map size) — just add more penalty terms.

I think we want two tier 4 pawns to feed a queen, 3 and 6 works. 5 + 2 as well.  pawn can survive on T - 1. it sounds like we want the tiers to be greater or equal to 7

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — that's a very clean rule system. Let me work through the algebra:

**The rule**: two pawns feed a queen when their **tier sum ≥ 7**

Each pawn on tier T exports: `h × T - d` (harvest × tier − diet)

Two pawns with tiers summing to 7: `h × 7 - 2d ≥ Q` (the binding case, e.g. T5+T2)

Pawn survives on T1: `h ≥ d`

The tightest case is `h = d`, giving `Q = 7h - 2h = 5h`:

| h = d | Q | Single T7 exports | T4+T4 exports | T5+T2 exports | T1+T5 exports |
|---|---|---|---|---|---|
| 1.0 | 5.0 | 6.0 ✅ | 6.0 ✅ | 5.0 ✅ | 4.0 ❌ |
| 1.2 | 6.0 | 7.2 ✅ | 7.2 ✅ | 6.0 ✅ | 4.8 ❌ |
| 1.4 | 7.0 | 8.4 ✅ | 8.4 ✅ | 7.0 ✅ | 5.6 ❌ |

All three rows satisfy every rule. The choice is just "how big are the numbers?" Let's verify with Optuna on the actual map:

```python
def pcr_profile_objective(trial):
    """Find cleanest profile satisfying the tier-sum-7 rule."""
    
    h = trial.suggest_float('pawn_harvest', 0.8, 2.0)
    d = trial.suggest_float('pawn_diet', 0.6, 2.0)
    Q = trial.suggest_float('queen_diet', 3.0, 8.0)
    
    penalty = 0
    
    # Rule 1: Two pawns with tier sum ≥ 7 feed the queen
    # Binding case: T5 + T2 = 7
    export_7 = h * 7 - 2 * d
    if export_7 < Q:
        penalty += 50 * (Q - export_7)
    
    # Rule 2: Tier sum 6 should NOT be enough (keeps it tight)
    export_6 = h * 6 - 2 * d
    if export_6 >= Q:
        penalty += 30 * (export_6 - Q + 0.1)
    
    # Rule 3: Pawn survives on T1
    if h < d:
        penalty += 40 * (d - h)
    
    # Rule 4: Single T7 pawn feeds the queen
    solo_t7 = h * 7 - d
    if solo_t7 < Q:
        penalty += 40 * (Q - solo_t7)
    
    # Rule 5: Queen/pawn diet ratio ≥ 3 (queen feels expensive)
    ratio = Q / max(d, 0.1)
    if ratio < 3.0:
        penalty += 10 * (3.0 - ratio)
    
    if penalty > 0:
        return 1000 + penalty
    
    # ── Run actual PCR on the map ──
    try:
        pcr = pawn_covered_route(
            start, finish,
            grid=myStuff.grid,
            elevations=myStuff.terr.elevations,
            food_tiers=fy.tiers,
            pawn_harvest=h, pawn_diet=d, queen_diet=Q,
            cone_range=4,
        )
    except Exception:
        return 999
    
    if pcr is None:
        return 998
    
    n_pawns = len(pcr['placements'])
    n_gaps = len(pcr['gaps'])
    
    # Prefer round numbers (small penalty for ugly decimals)
    roundness = sum(abs(v - round(v, 1)) for v in [h, d, Q])
    
    return n_pawns + 100 * n_gaps + 0.5 * roundness


# ── Run ──
import logging as _log
_log.getLogger('pyomo').setLevel(_log.ERROR)

study = optuna.create_study(direction='minimize', study_name='tier_sum_7')
study.optimize(pcr_profile_objective, n_trials=100, show_progress_bar=True)

best = study.best_trial
p = best.params
print(f"\nBest: {best.value:.1f} ({len([t for t in study.trials if t.value < 999])} feasible)")
print(f"  pawn_harvest: {p['pawn_harvest']:.2f}")
print(f"  pawn_diet:    {p['pawn_diet']:.2f}")
print(f"  queen_diet:   {p['queen_diet']:.2f}")

h, d, Q = p['pawn_harvest'], p['pawn_diet'], p['queen_diet']
print(f"\nVerification:")
for ta, tb in [(4,4), (3,6), (5,2), (1,5), (7,0)]:
    total = ta + tb
    if tb > 0:
        exp = h*(ta+tb) - 2*d
        label = f"T{ta}+T{tb}"
    else:
        exp = h*ta - d
        label = f"Solo T{ta}"
    ok = "✅" if exp >= Q else "❌"
    print(f"  {label:10s} sum={total} export={exp:.2f} vs Q={Q:.2f} {ok}")
print(f"  T1 survive: {h:.2f} ≥ {d:.2f} {'✅' if h >= d else '❌'}")
```

My guess: it'll converge near `h=d=1.2, Q=6.0` or `h=d=1.0, Q=5.0` — both satisfy the algebra and the roundness bonus will pull it to clean numbers. Then we take that profile and plug it back into the PCR + overlay to see the pawn positions on the map.

def pcr_profile_objective(trial):
    """Find cleanest profile satisfying the tier-sum-7 rule."""
    
    h = trial.suggest_float('pawn_harvest', 0.8, 2.0)
    d = trial.suggest_float('pawn_diet', 0.6, 2.0)
    Q = trial.suggest_float('queen_diet', 3.0, 8.0)
    
    penalty = 0
    
    # Rule 1: Two pawns with tier sum ≥ 7 feed the queen
    # Binding case: T5 + T2 = 7
    export_7 = h * 7 - 2 * d
    if export_7 < Q:
        penalty += 50 * (Q - export_7)
    
    # Rule 2: Tier sum 6 should NOT be enough (keeps it tight)
    export_6 = h * 6 - 2 * d
    if export_6 >= Q:
        penalty += 30 * (export_6 - Q + 0.1)
    
    # Rule 3: Pawn survives on T1
    if h < d:
        penalty += 40 * (d - h)
    
    # Rule 4: Single T7 pawn feeds the queen
    solo_t7 = h * 7 - d
    if solo_t7 < Q:
        penalty += 40 * (Q - solo_t7)
    
    # Rule 5: Queen/pawn diet ratio ≥ 3 (queen feels expensive)
    ratio = Q / max(d, 0.1)
    if ratio < 3.0:
        penalty += 10 * (3.0 - ratio)
    
    if penalty > 0:
        return 1000 + penalty
    
    # ── Run actual PCR on the map ──
    try:
        pcr = pawn_covered_route(
            start, finish,
            grid=myStuff.grid,
            elevations=myStuff.terr.elevations,
            food_tiers=fy.tiers,
            pawn_harvest=h, pawn_diet=d, queen_diet=Q,
            cone_range=4,
        )
    except Exception:
        return 999
    
    if pcr is None:
        return 998
    
    n_pawns = len(pcr['placements'])
    n_gaps = len(pcr['gaps'])
    
    # Prefer round numbers (small penalty for ugly decimals)
    roundness = sum(abs(v - round(v, 1)) for v in [h, d, Q])
    
    return n_pawns + 100 * n_gaps + 0.5 * roundness


# ── Run ──
import logging as _log
_log.getLogger('pyomo').setLevel(_log.ERROR)

study = optuna.create_study(direction='minimize', study_name='tier_sum_7')
study.optimize(pcr_profile_objective, n_trials=100, show_progress_bar=True)

best = study.best_trial
p = best.params
print(f"\nBest: {best.value:.1f} ({len([t for t in study.trials if t.value < 999])} feasible)")
print(f"  pawn_harvest: {p['pawn_harvest']:.2f}")
print(f"  pawn_diet:    {p['pawn_diet']:.2f}")
print(f"  queen_diet:   {p['queen_diet']:.2f}")

h, d, Q = p['pawn_harvest'], p['pawn_diet'], p['queen_diet']
print(f"\nVerification:")
for ta, tb in [(4,4), (3,6), (5,2), (1,5), (7,0)]:
    total = ta + tb
    if tb > 0:
        exp = h*(ta+tb) - 2*d
        label = f"T{ta}+T{tb}"
    else:
        exp = h*ta - d
        label = f"Solo T{ta}"
    ok = "✅" if exp >= Q else "❌"
    print(f"  {label:10s} sum={total} export={exp:.2f} vs Q={Q:.2f} {ok}")
print(f"  T1 survive: {h:.2f} ≥ {d:.2f} {'✅' if h >= d else '❌'}")


Here's the summary:

**Best profile found (trial 81, 10 pawns for 19-hex route):**

| Param | Value |
|---|---|
| `pawn_harvest` | **1.30** |
| `pawn_diet` | **0.70** |
| `queen_diet` | **6.86** |

**Tier-sum rule verification:**

| Combo | Sum | Export | Feeds queen? |
|---|---|---|---|
| T4+T4 | 8 | 8.99 | ✅ |
| T3+T6 | 9 | 10.29 | ✅ |
| T5+T2 | 7 | 7.69 | ✅ (barely) |
| T1+T5 | 6 | 6.39 | ❌ |
| Solo T7 | 7 | 8.39 | ✅ |
| T1 survive | — | 1.30 ≥ 0.70 | ✅ |

**The tier-sum-7 rule holds cleanly** — sum ≥ 7 works, sum 6 doesn't. The margin at sum=7 (T5+T2) is tight: 7.69 vs 6.86 = only 0.83 surplus. That's good game design — it works but there's no waste.

**Placement pattern** on the 19-hex route:
- T7 anchors at **both ends** (near settlements) — each solo-feeds the queen
- T2 backbone through the **middle** — these need to overlap in coverage
- T1 gap-fillers where needed (cheap pawns on bad land, tiny export)
- T5 hex covers the **late-middle** transition

**Key concern:** queen/pawn diet ratio is **9.8×** (6.86/0.70). That's extremely punishing — the original target was ≥ 3×. Pawns are almost free to feed. You might want to consider whether `pawn_diet=0.70` feels too low for gameplay, or if it's fine because the *quantity* of pawns (10 for a short route) is the real cost.

**Rounding suggestion:** `h=1.3, d=0.7, Q=7.0` gives nearly identical behavior with cleaner numbers (T5+T2 export = 7.7 vs Q=7.0 ✅, sum 6 = 6.4 ❌).

So these runs take up a bunch of context space. I am going to put the above run in a comment, but I want you to summarize the results so we can thing about it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's the summary:

**Best profile found (trial 81, 10 pawns for 19-hex route):**

| Param | Value |
|---|---|
| `pawn_harvest` | **1.30** |
| `pawn_diet` | **0.70** |
| `queen_diet` | **6.86** |

**Tier-sum rule verification:**

| Combo | Sum | Export | Feeds queen? |
|---|---|---|---|
| T4+T4 | 8 | 8.99 | ✅ |
| T3+T6 | 9 | 10.29 | ✅ |
| T5+T2 | 7 | 7.69 | ✅ (barely) |
| T1+T5 | 6 | 6.39 | ❌ |
| Solo T7 | 7 | 8.39 | ✅ |
| T1 survive | — | 1.30 ≥ 0.70 | ✅ |

**The tier-sum-7 rule holds cleanly** — sum ≥ 7 works, sum 6 doesn't. The margin at sum=7 (T5+T2) is tight: 7.69 vs 6.86 = only 0.83 surplus. That's good game design — it works but there's no waste.

**Placement pattern** on the 19-hex route:
- T7 anchors at **both ends** (near settlements) — each solo-feeds the queen
- T2 backbone through the **middle** — these need to overlap in coverage
- T1 gap-fillers where needed (cheap pawns on bad land, tiny export)
- T5 hex covers the **late-middle** transition

**Key concern:** queen/pawn diet ratio is **9.8×** (6.86/0.70). That's extremely punishing — the original target was ≥ 3×. Pawns are almost free to feed. You might want to consider whether `pawn_diet=0.70` feels too low for gameplay, or if it's fine because the *quantity* of pawns (10 for a short route) is the real cost.

**Rounding suggestion:** `h=1.3, d=0.7, Q=7.0` gives nearly identical behavior with cleaner numbers (T5+T2 export = 7.7 vs Q=7.0 ✅, sum 6 = 6.4 ❌).

Can you give a new apply profile with the rounding suggestion.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
FOOD_PROFILE_V2 = {
    'pawn_diet':       0.7,
    'pawn_harvest':    1.3,
    'pawn_capacity':  14.0,
    'pawn_sight':      4,
    'queen_diet':      7.0,
    'queen_harvest':   0.2,
    'queen_capacity': 24.0,
    'bishop_diet':     3.5,
    'bishop_harvest':  0.7,
    'bishop_capacity': 140.0,
}

def apply_food_profile_v2(pieces, profile=None):
    """Apply food profile to a list of pieces."""
    p = profile or FOOD_PROFILE_V2
    for piece in pieces:
        t = piece.piece_type.name.lower()
        for attr, field in [('diet','diet'), ('harvest','harvest_strength'),
                            ('capacity','food_capacity'), ('sight','sight')]:
            key = f'{t}_{attr}'
            if key in p:
                setattr(piece, field, p[key])
        # Fill to capacity
        piece.food = piece.food_capacity
```

Quick sanity check you can run after:

```python
p = FOOD_PROFILE_V2
h, d, Q = p['pawn_harvest'], p['pawn_diet'], p['queen_diet']

print("Tier-sum rule (2 pawns, need sum ≥ 7):")
for ta, tb in [(4,4), (3,6), (5,2), (1,5), (3,4), (7,0)]:
    s = ta + tb
    exp = h*s - 2*d if tb > 0 else h*ta - d
    label = f"T{ta}+T{tb}" if tb > 0 else f"Solo T{ta}"
    print(f"  {label:10s} sum={s:2d}  export={exp:.1f}  {'✅' if exp >= Q else '❌'}")
print(f"\n  Pawn on T1: {h*1:.1f} vs diet {d:.1f} → {'survives' if h >= d else 'starves'}")
print(f"  Queen/pawn ratio: {Q/d:.1f}×")
```

Expected output:
- T4+T4 (sum 8): 9.0 ✅
- T5+T2 (sum 7): 7.7 ✅
- T3+T4 (sum 7): 7.7 ✅
- T1+T5 (sum 6): 6.4 ❌
- Solo T7: 8.4 ✅
- Pawn survives T1, ratio 10×

In [ ]:
FOOD_PROFILE_V2 = {
    'pawn_diet':       0.7,
    'pawn_harvest':    1.3,
    'pawn_capacity':  14.0,
    'pawn_sight':      4,
    'queen_diet':      7.0,
    'queen_harvest':   0.2,
    'queen_capacity': 24.0,
    'bishop_diet':     3.5,
    'bishop_harvest':  0.7,
    'bishop_capacity': 140.0,
}

def apply_food_profile_v2(pieces, profile=None):
    """Apply food profile to a list of pieces."""
    p = profile or FOOD_PROFILE_V2
    for piece in pieces:
        t = piece.piece_type.name.lower()
        for attr, field in [('diet','diet'), ('harvest','harvest_strength'),
                            ('capacity','food_capacity'), ('sight','sight')]:
            key = f'{t}_{attr}'
            if key in p:
                setattr(piece, field, p[key])
        # Fill to capacity
        piece.food = piece.food_capacity


In [ ]:
p = FOOD_PROFILE_V2
h, d, Q = p['pawn_harvest'], p['pawn_diet'], p['queen_diet']

print("Tier-sum rule (2 pawns, need sum ≥ 7):")
for ta, tb in [(4,4), (3,6), (5,2), (1,5), (3,4), (7,0)]:
    s = ta + tb
    exp = h*s - 2*d if tb > 0 else h*ta - d
    label = f"T{ta}+T{tb}" if tb > 0 else f"Solo T{ta}"
    print(f"  {label:10s} sum={s:2d}  export={exp:.1f}  {'✅' if exp >= Q else '❌'}")
print(f"\n  Pawn on T1: {h*1:.1f} vs diet {d:.1f} → {'survives' if h >= d else 'starves'}")
print(f"  Queen/pawn ratio: {Q/d:.1f}×")


Looks good?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looks great! Clean results:

- **Tier-sum ≥ 7 rule holds perfectly** — every sum-7 combo passes, sum-6 fails
- **Tight margin at sum=7** — only 0.7 surplus (7.7 vs 7.0), so it works but there's no slack. Good tension.
- **Solo T7 pawn feeds the queen** — strategic chokepoint mechanic works
- **Pawn survives on T1** — pawns never starve, even on the worst land
- **10× diet ratio** — queen is very expensive, which makes escort logistics the core game

The numbers are all round and easy for a player to reason about: *"I need my two pawns' tiers to add up to 7."* That's a great mental model.

One thing to note: we don't have knight/rook profiles in V2 yet. Worth adding those when you're ready, but for now this is solid for testing the PCR. Want to rerun the route with this profile and get the overlay working?

Can you rerun the route with this profile and then simulate the queen going between settlements?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's do it in two steps. First, find the pawn placements:

```python
rebootWorld()

# Apply V2 profile to all pieces
all_pieces = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        all_pieces.extend(city.citizens)
apply_food_profile_v2(all_pieces)

fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)

start = cities[0].location
finish = cities[1].location
print(f"Route: ⬡{start} → ⬡{finish}")

pcr = pawn_covered_route(
    start, finish,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pawn_harvest=FOOD_PROFILE_V2['pawn_harvest'],
    pawn_diet=FOOD_PROFILE_V2['pawn_diet'],
    queen_diet=FOOD_PROFILE_V2['queen_diet'],
    cone_range=FOOD_PROFILE_V2['pawn_sight'],
)
```

Run that and check the output — we should see the placements and coverage. Then simulate the queen walking the route:

```python
def simulate_queen_march(pcr, queen, grid, elevations, food_tiers, profile=None):
    """Simulate the queen walking the PCR route, fed by placed pawns.
    
    Each turn:
      1. Queen moves to next route hex
      2. Queen eats (diet deducted from food)
      3. All pawns in cone_range harvest and give surplus to queen
      4. Record state
    
    Returns DataFrame of the march log.
    """
    p = profile or FOOD_PROFILE_V2
    route = pcr['route']
    placements = pcr['placements']
    cone = p['pawn_sight']
    
    # Build pawn positions + stats
    pawns = []
    for pl in placements:
        pawns.append({
            'hex': pl['hex'], 'tier': pl['tier'],
            'food': p['pawn_capacity'],
            'export': p['pawn_harvest'] * pl['tier'] - p['pawn_diet'],
        })
    
    queen_food = p['queen_capacity']
    queen_hp = queen.health
    log = []
    
    for turn, hex_idx in enumerate(route):
        # ── Queen eats ──
        queen_food -= p['queen_diet']
        
        # ── Pawns in range harvest & give ──
        hp_q = grid.index_to_hexposition(hex_idx)
        feed_this_turn = 0
        feeders = 0
        for pawn in pawns:
            hp_p = grid.index_to_hexposition(pawn['hex'])
            if hp_p.distance(hp_q) <= cone and pawn['export'] > 0:
                give = pawn['export']
                queen_food += give
                feed_this_turn += give
                feeders += 1
                # Pawn also eats (already accounted in export = harvest*tier - diet)
        
        # Clamp to capacity
        queen_food = min(queen_food, p['queen_capacity'])
        
        # ── Health damage if starving ──
        if queen_food < 0:
            queen_hp += queen_food * 5  # lose 5 hp per unit of starvation
            queen_food = 0
        
        log.append({
            'turn': turn,
            'hex': hex_idx,
            'queen_food': round(queen_food, 2),
            'queen_hp': round(queen_hp, 1),
            'feeders': feeders,
            'feed_rate': round(feed_this_turn, 2),
            'surplus': round(feed_this_turn - p['queen_diet'], 2),
        })
        
        if queen_hp <= 0:
            print(f"💀 Queen died on turn {turn} at ⬡{hex_idx}")
            break
    
    df = pd.DataFrame(log)
    
    # ── Summary ──
    alive = queen_hp > 0
    min_food = df['queen_food'].min()
    uncovered = df[df['feeders'] == 0]
    
    status = "✅ SURVIVED" if alive else "💀 DIED"
    print(f"\n{status}: {len(route)} hexes in {len(df)} turns")
    print(f"  Food: min={min_food:.1f}, final={df.iloc[-1]['queen_food']:.1f}")
    print(f"  HP: {queen_hp:.0f}/{queen.max_health}")
    print(f"  Uncovered turns: {len(uncovered)}/{len(df)}")
    if len(uncovered) > 0:
        print(f"  Gap hexes: {uncovered['hex'].tolist()}")
    
    return df


# ── Run it ──
queen = myStuff.queen
queen.food = FOOD_PROFILE_V2['queen_capacity']

march_log = simulate_queen_march(pcr, queen, myStuff.grid,
                                  myStuff.terr.elevations, fy.tiers)
print(march_log.to_string(index=False))
```

The log shows food level each turn — you'll see it dip in gaps and recover when pawns are in range. If the queen dies we'll know exactly which hex was the problem and can look at whether it's a coverage gap or a tier issue.

In [ ]:
rebootWorld()

# Apply V2 profile to all pieces
all_pieces = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        all_pieces.extend(city.citizens)
apply_food_profile_v2(all_pieces)

fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

cities = []
for country in myStuff.board.kingdoms:
    for city in country.settlements:
        cities.append(city)

start = cities[0].location
finish = cities[1].location
print(f"Route: ⬡{start} → ⬡{finish}")

pcr = pawn_covered_route(
    start, finish,
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pawn_harvest=FOOD_PROFILE_V2['pawn_harvest'],
    pawn_diet=FOOD_PROFILE_V2['pawn_diet'],
    queen_diet=FOOD_PROFILE_V2['queen_diet'],
    cone_range=FOOD_PROFILE_V2['pawn_sight'],
)


In [ ]:
def simulate_queen_march(pcr, queen, grid, elevations, food_tiers, profile=None):
    """Simulate the queen walking the PCR route, fed by placed pawns.
    
    Each turn:
      1. Queen moves to next route hex
      2. Queen eats (diet deducted from food)
      3. All pawns in cone_range harvest and give surplus to queen
      4. Record state
    
    Returns DataFrame of the march log.
    """
    p = profile or FOOD_PROFILE_V2
    route = pcr['route']
    placements = pcr['placements']
    cone = p['pawn_sight']
    
    # Build pawn positions + stats
    pawns = []
    for pl in placements:
        pawns.append({
            'hex': pl['hex'], 'tier': pl['tier'],
            'food': p['pawn_capacity'],
            'export': p['pawn_harvest'] * pl['tier'] - p['pawn_diet'],
        })
    
    queen_food = p['queen_capacity']
    queen_hp = queen.health
    log = []
    
    for turn, hex_idx in enumerate(route):
        # ── Queen eats ──
        queen_food -= p['queen_diet']
        
        # ── Pawns in range harvest & give ──
        hp_q = grid.index_to_hexposition(hex_idx)
        feed_this_turn = 0
        feeders = 0
        for pawn in pawns:
            hp_p = grid.index_to_hexposition(pawn['hex'])
            if hp_p.distance(hp_q) <= cone and pawn['export'] > 0:
                give = pawn['export']
                queen_food += give
                feed_this_turn += give
                feeders += 1
                # Pawn also eats (already accounted in export = harvest*tier - diet)
        
        # Clamp to capacity
        queen_food = min(queen_food, p['queen_capacity'])
        
        # ── Health damage if starving ──
        if queen_food < 0:
            queen_hp += queen_food * 5  # lose 5 hp per unit of starvation
            queen_food = 0
        
        log.append({
            'turn': turn,
            'hex': hex_idx,
            'queen_food': round(queen_food, 2),
            'queen_hp': round(queen_hp, 1),
            'feeders': feeders,
            'feed_rate': round(feed_this_turn, 2),
            'surplus': round(feed_this_turn - p['queen_diet'], 2),
        })
        
        if queen_hp <= 0:
            print(f"💀 Queen died on turn {turn} at ⬡{hex_idx}")
            break
    
    df = pd.DataFrame(log)
    
    # ── Summary ──
    alive = queen_hp > 0
    min_food = df['queen_food'].min()
    uncovered = df[df['feeders'] == 0]
    
    status = "✅ SURVIVED" if alive else "💀 DIED"
    print(f"\n{status}: {len(route)} hexes in {len(df)} turns")
    print(f"  Food: min={min_food:.1f}, final={df.iloc[-1]['queen_food']:.1f}")
    print(f"  HP: {queen_hp:.0f}/{queen.max_health}")
    print(f"  Uncovered turns: {len(uncovered)}/{len(df)}")
    if len(uncovered) > 0:
        print(f"  Gap hexes: {uncovered['hex'].tolist()}")
    
    return df


# ── Run it ──
queen = myStuff.queen
queen.food = FOOD_PROFILE_V2['queen_capacity']

march_log = simulate_queen_march(pcr, queen, myStuff.grid,
                                  myStuff.terr.elevations, fy.tiers)
print(march_log.to_string(index=False))


now try with the food simulator. assign Harvest share to the pawns and have the queen go.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good — this is the real test. We need to create actual pawn pieces at the PCR positions, give them harvest/give orders, and send the queen along the route through the real simulator.

```python
# ── Step 1: Create pawn pieces at PCR placements ──
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList

# Fresh squad for the PCR
pcr_squad = Squad.squads(1, myStuff.country.flag)[0]

# Add the queen
queen = myStuff.queen
queen.location = pcr['route'][0]  # start of route
queen.food = FOOD_PROFILE_V2['queen_capacity']
pcr_squad.pieces.append(queen)

# Create pawns at each placement hex
for i, pl in enumerate(pcr['placements']):
    pawn = myStuff.home.recruit(PieceType.PAWN, flag=pcr_squad.flag)
    pawn.location = pl['hex']
    
    # Face toward nearest route hex
    hp_p = myStuff.grid.index_to_hexposition(pl['hex'])
    mid_w = sorted(pl['coverage'])[len(pl['coverage']) // 2]
    route_hex = pcr['route'][mid_w]
    pawn.facing = _facing_toward(myStuff.grid, pl['hex'], route_hex)
    
    # Instructions: harvest then give, on patrol (loop forever)
    pawn.instructions = InstructionList(
        [Instruction.HARVEST.value, Instruction.GIVE.value],
        cursor=0, patrol=True)
    
    pcr_squad.pieces.append(pawn)

# Apply V2 profile to all pieces in the squad
apply_food_profile_v2(pcr_squad.pieces)

print(f"Squad: {len(pcr_squad.pieces)} pieces "
      f"({len(pcr_squad.alive)} alive)")
for p in pcr_squad.pieces:
    print(f"  {p.name:10s} {p.piece_type.name:6s} ⬡{p.location} "
          f"facing={p.facing} food={p.food:.1f}")
```

```python
# ── Step 2: Queen movement instructions along the route ──
# Pathfind gives us the proper instruction sequence (FORWARD, ROT_L, ROT_R)
tp = queen.pathfind(pcr['route'][-1], myStuff.grid,
                    myStuff.terr.elevations,
                    myStuff.terr.fields.get('country'))

if tp:
    rules = tp.to_rules(myStuff.grid, queen.facing)
    queen.instructions = InstructionList(rules, cursor=0, patrol=False)
    print(f"Queen instructions: {len(rules)} steps")
    print(f"  Route: {len(pcr['route'])} hexes")
else:
    print("⚠️ Queen pathfind failed — building manual route")
```

```python
# ── Step 3: Run the simulator ──
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=pcr_squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[pcr_squad],
    callbacks={pcr_squad.id: food_redistribute_adjacent},
)

sim.run(num_turns=30)
sim.summary()
```

```python
# ── Step 4: Check the queen's journey ──
queen_log = [(r.turn, queen.location, queen.food, queen.health)
             for r in sim.reports]
df = pd.DataFrame(queen_log, columns=['turn', 'hex', 'food', 'hp'])
print(df.to_string(index=False))
print(f"\nQueen arrived: {'✅' if queen.location == pcr['route'][-1] else '❌'}")
print(f"Final food: {queen.food:.1f}, HP: {queen.health}/{queen.max_health}")
```

Run these blocks in order. Key differences from the simple sim:

- **Real movement**: queen actually pathfinds, rotates, steps — costs turns
- **Real GIVE mechanics**: pawns must face the queen, range matters
- **`food_redistribute_adjacent`**: tier-1 sharing kicks in automatically (no instruction edits, safe during march)
- **Real hunger**: if GIVE doesn't reach, the queen actually starves

If the queen runs out of food mid-route, we'll see exactly which turn and hex — then we can diagnose whether it's a facing issue (pawn looking the wrong way), a range issue (queen moved out of cone), or a coverage gap.

In [ ]:
# ── Step 1: Create pawn pieces at PCR placements ──
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList

# Fresh squad for the PCR
pcr_squad = Squad.squads(1, myStuff.country.flag)[0]

# Add the queen
queen = myStuff.queen
queen.location = pcr['route'][0]  # start of route
queen.food = FOOD_PROFILE_V2['queen_capacity']
pcr_squad.pieces.append(queen)

# Create pawns at each placement hex
for i, pl in enumerate(pcr['placements']):
    pawn = myStuff.home.recruit(PieceType.PAWN, flag=pcr_squad.flag)
    pawn.location = pl['hex']
    
    # Face toward nearest route hex
    hp_p = myStuff.grid.index_to_hexposition(pl['hex'])
    mid_w = sorted(pl['coverage'])[len(pl['coverage']) // 2]
    route_hex = pcr['route'][mid_w]
    pawn.facing = _facing_toward(myStuff.grid, pl['hex'], route_hex)
    
    # Instructions: harvest then give, on patrol (loop forever)
    pawn.instructions = InstructionList(
        [Instruction.HARVEST.value, Instruction.GIVE.value],
        cursor=0, patrol=True)
    
    pcr_squad.pieces.append(pawn)

# Apply V2 profile to all pieces in the squad
apply_food_profile_v2(pcr_squad.pieces)

print(f"Squad: {len(pcr_squad.pieces)} pieces "
      f"({len(pcr_squad.alive)} alive)")
for p in pcr_squad.pieces:
    print(f"  {p.name:10s} {p.piece_type.name:6s} ⬡{p.location} "
          f"facing={p.facing} food={p.food:.1f}")


In [ ]:
# ── Step 2: Queen movement instructions along the route ──
# Pathfind gives us the proper instruction sequence (FORWARD, ROT_L, ROT_R)
tp = queen.pathfind(pcr['route'][-1], myStuff.grid,
                    myStuff.terr.elevations,
                    myStuff.terr.fields.get('country'))

if tp:
    rules = tp.to_rules(myStuff.grid, queen.facing)
    queen.instructions = InstructionList(rules, cursor=0, patrol=False)
    print(f"Queen instructions: {len(rules)} steps")
    print(f"  Route: {len(pcr['route'])} hexes")
else:
    print("⚠️ Queen pathfind failed — building manual route")


In [ ]:
# ── Step 3: Run the simulator ──
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=pcr_squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[pcr_squad],
    callbacks={pcr_squad.id: food_redistribute_adjacent},
)

sim.run(num_turns=30)
sim.summary()


In [ ]:
show(sim)

In [ ]:
# ── Step 4: Check the queen's journey ──
queen_id = queen.id

rows = []
for snap in sim.snapshots:
    qs = snap['piece_states'].get(queen_id)
    if qs:
        rows.append({
            'turn': snap['turn'],
            'hex': qs['location'],
            'food': round(qs['food'], 2),
            'hp': round(qs['health'], 1),
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print(f"\nQueen arrived: {'✅' if queen.location == pcr['route'][-1] else '❌'}")
print(f"Final food: {queen.food:.1f}, HP: {queen.health}/{queen.max_health}")

# How many gives did the queen receive?
gives = sim.df[(sim.df['event'] == 'gave') & (sim.df['dst_piece_id'] == queen_id)]
print(f"Total gives received: {len(gives)}, total food: {gives['amount'].sum():.1f}")


In [ ]:
??FoodSimulator

In [ ]:
??FoodSimulator.__ft__

did the queen move at all?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

She moved for the first 6 turns — `158→217→256→274→273→332→329→327` — then got stuck at ⬡327 for the remaining 24 turns. The summary says "16 hexes" total, so she was moving multiple hexes per turn initially (she has movement budget > 1), but then stalled.

Two problems:

1. **Off-route**: Her actual path (`217→256→274→273→332→329→327`) doesn't match the PCR route (`158→178→197→217→236→256→275→295→...→345`). The `pathfind` method found a different path than the BFS in `pawn_covered_route`, so she wandered away from the pawns.

2. **Stuck at 327**: She ran out of instructions or hit a blocked hex. After 24 instructions were consumed she had nowhere to go.

The fix is to make the queen's instructions match the PCR route exactly, not use a separate pathfind:

```python
# Build queen instructions from the PCR route directly
from HexMagic.game.data import Instruction, InstructionList

def route_to_instructions(route, grid, start_facing):
    """Convert a list of hex indices into FORWARD/ROT_L/ROT_R instructions."""
    rules = []
    facing = start_facing
    for i in range(len(route) - 1):
        # Which direction is the next hex?
        target_facing = _facing_toward(grid, route[i], route[i + 1])
        # Rotate to face it
        while facing != target_facing:
            diff = (target_facing - facing) % 6
            if diff <= 3:
                rules.append(Instruction.ROT_R.value)
                facing = (facing + 1) % 6
            else:
                rules.append(Instruction.ROT_L.value)
                facing = (facing - 1) % 6
        rules.append(Instruction.FORWARD.value)
    return rules, facing

rules, _ = route_to_instructions(pcr['route'], myStuff.grid, queen.facing)
queen.instructions = InstructionList(rules, cursor=0, patrol=False)
print(f"Route instructions: {len(rules)} steps for {len(pcr['route'])} hexes")
```

This guarantees the queen follows the exact BFS route that the pawns were placed to cover, rather than taking a different shortest path.

## pass 2

@patch
def _try_forward(self: FoodSimulator, piece, occupied, slot=None, budget=float('inf')):
    d = HexPosition.directions()[piece.facing % 6]
    nbr = self.grid.hexposition_to_index(d, piece.location)
    
    passable = (0 <= nbr < len(self.elevations)
                and nbr not in self.grid.invalidRegion
                and self.elevations[nbr] >= 1)
    
    if not passable:
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 0.0
    
    if nbr in occupied and nbr != piece.location:
        blocker = next((p for p in self.alive if p.location == nbr), None)
        if blocker:
            bump_cost = self._try_bump(piece, blocker, occupied, slot)
            if bump_cost is not None:
                return bump_cost
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 0.0
    
    cost = _move_cost(self.elevations, piece.location, nbr)
    if cost > budget:
        return 0.0  # Can't afford — don't move!
    
    src = piece.location
    occupied.discard(src); piece.location = nbr; occupied.add(nbr)
    self._log(piece, SimEventType.MOVED, slot=slot, src_hex=src, dst_hex=nbr, amount=cost)
    return cost


@patch
def _tick_one(self: FoodSimulator, piece, budget, occupied, slot=None):
    il = piece.instructions
    if not il.rules: return None, 0.0
    while True:
        if il.cursor >= len(il.rules):
            if il.patrol: il.cursor = 0
            else: return None, 0.0
        instr = Instruction(il.rules[il.cursor])
        if instr in _FREE:
            piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                            else (piece.facing + 1)) % 6
            self._log(piece, SimEventType.ROTATED, slot=slot)
            il.cursor += 1; continue
        if instr in _COSTS_ONE:
            if budget < 1.0: return None, 0.0
            il.cursor += 1
            if instr == Instruction.HARVEST: self._do_harvest(piece, slot)
            elif instr == Instruction.GIVE: self._do_give(piece, slot)
            return instr, 1.0
        # FORWARD — pass budget so _try_forward won't move if we can't afford it
        if budget < 1.0: return None, 0.0
        cost = self._try_forward(piece, occupied, slot, budget=budget)
        if cost > 0:
            il.cursor += 1
            return Instruction.FORWARD, cost
        return None, 0.0


In [ ]:
# ── Step 1: Create pawn pieces at PCR placements ──
from HexMagic.game.data import Piece, PieceType, Instruction, InstructionList

# Fresh squad for the PCR
pcr_squad = Squad.squads(1, myStuff.country.flag)[0]

# Add the queen
queen = myStuff.queen
queen.location = pcr['route'][0]  # start of route
queen.food = FOOD_PROFILE_V2['queen_capacity']
pcr_squad.pieces.append(queen)

# Create pawns at each placement hex
for i, pl in enumerate(pcr['placements']):
    pawn = myStuff.home.recruit(PieceType.PAWN, flag=pcr_squad.flag)
    pawn.location = pl['hex']
    
    # Face toward nearest route hex
    hp_p = myStuff.grid.index_to_hexposition(pl['hex'])
    mid_w = sorted(pl['coverage'])[len(pl['coverage']) // 2]
    route_hex = pcr['route'][mid_w]
    pawn.facing = _facing_toward(myStuff.grid, pl['hex'], route_hex)
    
    # Instructions: harvest then give, on patrol (loop forever)
    pawn.instructions = InstructionList(
        [Instruction.HARVEST.value, Instruction.GIVE.value],
        cursor=0, patrol=True)
    
    pcr_squad.pieces.append(pawn)

# Apply V2 profile to all pieces in the squad
apply_food_profile_v2(pcr_squad.pieces)

print(f"Squad: {len(pcr_squad.pieces)} pieces "
      f"({len(pcr_squad.alive)} alive)")
for p in pcr_squad.pieces:
    print(f"  {p.name:10s} {p.piece_type.name:6s} ⬡{p.location} "
          f"facing={p.facing} food={p.food:.1f}")


In [ ]:
# Build queen instructions from the PCR route directly
from HexMagic.game.data import Instruction, InstructionList

def route_to_instructions(route, grid, start_facing):
    """Convert a list of hex indices into FORWARD/ROT_L/ROT_R instructions."""
    rules = []
    facing = start_facing
    for i in range(len(route) - 1):
        # Which direction is the next hex?
        target_facing = _facing_toward(grid, route[i], route[i + 1])
        # Rotate to face it
        while facing != target_facing:
            diff = (target_facing - facing) % 6
            if diff <= 3:
                rules.append(Instruction.ROT_R.value)
                facing = (facing + 1) % 6
            else:
                rules.append(Instruction.ROT_L.value)
                facing = (facing - 1) % 6
        rules.append(Instruction.FORWARD.value)
    return rules, facing

rules, _ = route_to_instructions(pcr['route'], myStuff.grid, queen.facing)
queen.instructions = InstructionList(rules, cursor=0, patrol=False)
print(f"Route instructions: {len(rules)} steps for {len(pcr['route'])} hexes")


In [ ]:
def route_to_instructions(route, grid, start_facing):
    """Convert a list of adjacent hex indices into exact FORWARD/ROT instructions."""
    rules = []
    facing = start_facing
    dirs = HexPosition.directions()
    
    for i in range(len(route) - 1):
        # Which of the 6 directions goes from route[i] to route[i+1]?
        target_facing = None
        for d in range(6):
            nb_idx = grid.hexposition_to_index(dirs[d], route[i])
            if nb_idx == route[i + 1]:
                target_facing = d
                break
        
        if target_facing is None:
            print(f"⚠️ {route[i]}→{route[i+1]} not adjacent! Skipping.")
            continue
        
        # Rotate shortest direction
        while facing != target_facing:
            diff = (target_facing - facing) % 6
            if diff <= 3:
                rules.append(Instruction.ROT_R.value)
                facing = (facing + 1) % 6
            else:
                rules.append(Instruction.ROT_L.value)
                facing = (facing - 1) % 6
        
        rules.append(Instruction.FORWARD.value)
    return rules, facing

rules, _ = route_to_instructions(pcr['route'], myStuff.grid, queen.facing)
queen.instructions = InstructionList(rules, cursor=0, patrol=False)
print(f"Route instructions: {len(rules)} steps for {len(pcr['route'])} hexes")


In [ ]:
grid = myStuff.grid 
nRows = grid.nRows
nCols =  grid.nCols

i = 0
for row in range(nRows):

    line = "|"
    for col in range(nCols):
        line += f" { int(i):3}"
        i += 1

    print(line + " |")

In [ ]:
!cat ../../../HexMagic/plot/*.py

So the adjacency is wrong. 352 and 351 are adjacent.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Right — the bug is in how we're looking up neighbors. We're mixing up coordinate origins. The fix is simple: use `HexPosition.directions()` directly as offsets from `route[i]`:

```python
def route_to_instructions(route, grid, start_facing):
    """Convert a list of adjacent hex indices into exact FORWARD/ROT instructions."""
    rules = []
    facing = start_facing
    dirs = HexPosition.directions()
    
    for i in range(len(route) - 1):
        # Which of the 6 directions goes from route[i] to route[i+1]?
        target_facing = None
        for d in range(6):
            nb_idx = grid.hexposition_to_index(dirs[d], route[i])
            if nb_idx == route[i + 1]:
                target_facing = d
                break
        
        if target_facing is None:
            print(f"⚠️ {route[i]}→{route[i+1]} not adjacent! Skipping.")
            continue
        
        # Rotate shortest direction
        while facing != target_facing:
            diff = (target_facing - facing) % 6
            if diff <= 3:
                rules.append(Instruction.ROT_R.value)
                facing = (facing + 1) % 6
            else:
                rules.append(Instruction.ROT_L.value)
                facing = (facing - 1) % 6
        
        rules.append(Instruction.FORWARD.value)
    return rules, facing

rules, _ = route_to_instructions(pcr['route'], myStuff.grid, queen.facing)
queen.instructions = InstructionList(rules, cursor=0, patrol=False)
print(f"Route instructions: {len(rules)} steps for {len(pcr['route'])} hexes")
```

The original code was calling `hp.neighbor(d)` which gives a position relative to the **grid middle**, then passing `route[i]` as the origin to `hexposition_to_index` — mismatched origins. Using `dirs[d]` (the unit direction vector) with `route[i]` as the origin means "what's one step in direction d from here?" — which is exactly what we want.

In [ ]:
# ── Step 3: Run the simulator ──
fy = FoodYield(myStuff.terr, myStuff.basin); fy.compute()

sim = FoodSimulator(
    grid=myStuff.grid,
    elevations=myStuff.terr.elevations,
    food_tiers=fy.tiers,
    pieces=pcr_squad.alive,
    countries=myStuff.terr.fields.get('country'),
    squads=[pcr_squad],
    callbacks={pcr_squad.id: food_redistribute_adjacent},
)

sim.run(num_turns=30)
sim.summary()


In [ ]:
# ── Step 4: Check the queen's journey ──
queen_id = queen.id

rows = []
for snap in sim.snapshots:
    qs = snap['piece_states'].get(queen_id)
    if qs:
        rows.append({
            'turn': snap['turn'],
            'hex': qs['location'],
            'food': round(qs['food'], 2),
            'hp': round(qs['health'], 1),
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

print(f"\nQueen arrived: {'✅' if queen.location == pcr['route'][-1] else '❌'}")
print(f"Final food: {queen.food:.1f}, HP: {queen.health}/{queen.max_health}")

# How many gives did the queen receive?
gives = sim.df[(sim.df['event'] == 'gave') & (sim.df['dst_piece_id'] == queen_id)]
print(f"Total gives received: {len(gives)}, total food: {gives['amount'].sum():.1f}")


she is getting stuck?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — two issues. First, she's moving multiple hexes per turn (movement budget > 1), so she burns through 20 instructions in ~7 turns. Second, she's going **off-route** — at ⬡334 she went to ⬡373 instead of ⬡353. That means `_facing_toward` is computing an approximate direction that lands on the wrong neighbor.

The fix: instead of using `_facing_toward` (angle-based approximation), find the **exact** neighbor direction:

```python
def route_to_instructions(route, grid, start_facing):
    """Convert a list of adjacent hex indices into exact FORWARD/ROT instructions."""
    rules = []
    facing = start_facing
    for i in range(len(route) - 1):
        # Find exact direction: which of the 6 neighbors is route[i+1]?
        target_facing = None
        hp = grid.index_to_hexposition(route[i])
        for d in range(6):
            nb_hp = hp.neighbor(d)
            nb_idx = grid.hexposition_to_index(nb_hp, route[i])
            if nb_idx == route[i + 1]:
                target_facing = d
                break
        
        if target_facing is None:
            print(f"⚠️ {route[i]}→{route[i+1]} not adjacent! Skipping.")
            continue
        
        # Rotate to face it (shortest direction)
        while facing != target_facing:
            diff = (target_facing - facing) % 6
            if diff <= 3:
                rules.append(Instruction.ROT_R.value)
                facing = (facing + 1) % 6
            else:
                rules.append(Instruction.ROT_L.value)
                facing = (facing - 1) % 6
        
        rules.append(Instruction.FORWARD.value)
    return rules, facing
```

The key difference: instead of `_facing_toward(grid, A, B)` doing trig on hex centers, this checks all 6 neighbors and finds the **exact** direction `d` where `neighbor(d) == next_hex`. No angle approximation = no wrong neighbor.

If you still see her stalling after the instructions run out (because she crosses them in fewer turns than expected), the route is correct but she's just fast. That's fine — she'll arrive early and sit at the destination.

Is she going off route?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — the PCR route is `158→178→197→217→236→256→275→295→314→334→353→352→351→...→345`, but she went `334→373` instead of `334→353`. That's one row too far south.

The bug is almost certainly a **direction convention mismatch**: `route_to_instructions` uses `HexPosition.directions()` which maps indices 0–5 as `[SW, W, NW, NE, E, SE]`, but the piece's `facing` attribute might use a different ordering (e.g. the game engine's forward movement maps facing=0 to a different direction).

Let's verify:

```python
# Debug: what direction does the game think each facing goes?
grid = myStuff.grid
test_hex = 334
print(f"From ⬡{test_hex}, neighbors by HexPosition direction:")
dirs = HexPosition.directions()
for d in range(6):
    nb = grid.hexposition_to_index(dirs[d], test_hex)
    print(f"  dir {d} ({dirs[d].desc():3s}): ⬡{nb}")

print(f"\nFrom ⬡{test_hex}, neighbors by piece.facing forward:")
for f in range(6):
    queen.location = test_hex
    queen.facing = f
    # Where would FORWARD take us?
    fwd_hp = dirs[f]  # this is what route_to_instructions assumes
    fwd_idx = grid.hexposition_to_index(fwd_hp, test_hex)
    print(f"  facing {f}: ⬡{fwd_idx}")

print(f"\nRoute expects: 334 → 353")
print(f"Queen went:    334 → 373")
```

This will show us if direction index 0 in `HexPosition.directions()` maps to the same hex as `facing=0` in the game engine. If they don't match, we need a remapping table between the two conventions.

In [ ]:
# Debug: what direction does the game think each facing goes?
grid = myStuff.grid
test_hex = 334
print(f"From ⬡{test_hex}, neighbors by HexPosition direction:")
dirs = HexPosition.directions()
for d in range(6):
    nb = grid.hexposition_to_index(dirs[d], test_hex)
    print(f"  dir {d} ({dirs[d].desc():3s}): ⬡{nb}")

print(f"\nFrom ⬡{test_hex}, neighbors by piece.facing forward:")
for f in range(6):
    queen.location = test_hex
    queen.facing = f
    # Where would FORWARD take us?
    fwd_hp = dirs[f]  # this is what route_to_instructions assumes
    fwd_idx = grid.hexposition_to_index(fwd_hp, test_hex)
    print(f"  facing {f}: ⬡{fwd_idx}")

print(f"\nRoute expects: 334 → 353")
print(f"Queen went:    334 → 373")


Found the bug?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The **directions match perfectly** — that's not the bug. Facing 0 = SW = ⬡353 from ⬡334, exactly correct.

The bug is subtle: she goes `334→353→373` — that's **two** SW steps, then later she goes west on row 18 (`373→372→371→370→369`). The ROT_R *did* eventually execute (she turned west), but **one FORWARD too late**.

Let's verify the actual instruction list:

```python
# What's actually in the instruction list?
print(f"Total instructions: {len(queen.instructions.rules)}")
for i, r in enumerate(queen.instructions.rules):
    instr = Instruction(r)
    print(f"  [{i:2d}] {instr.name:10s} (value={r})")

# Index 11 should be ROT_R — is it?
print(f"\nCritical check: instruction[11] = {Instruction(queen.instructions.rules[11]).name}")
```

I suspect the instruction at index 11 is FORWARD when it should be ROT_R. If so, the bug is in `route_to_instructions` miscounting — possibly `hexposition_to_index(dirs[d], route[i])` returns a wrong neighbor for one specific row parity, causing the `target_facing` lookup to fail for the 353→352 transition and defaulting to the wrong thing.

In [ ]:
# What's actually in the instruction list?
print(f"Total instructions: {len(queen.instructions.rules)}")
for i, r in enumerate(queen.instructions.rules):
    instr = Instruction(r)
    print(f"  [{i:2d}] {instr.name:10s} (value={r})")

# Index 11 should be ROT_R — is it?
print(f"\nCritical check: instruction[11] = {Instruction(queen.instructions.rules[11]).name}")


thoughts?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The instructions look correct (10 SW, ROT_R, 8 W), but she goes 334→353→**373** instead of 334→353→ROT_R→**352**. She takes an extra SW step past the turn.

Then she **gets blocked by pawn Evelyn at ⬡368** — since she's off-route (row 18 instead of row 17), her W path collides with the pawn placement.

Let's dump the event log to see exactly what happened:

```python
# What did the queen actually do each turn?
q_events = sim.df[sim.df['piece_id'] == queen_id].copy()
q_events = q_events[q_events['event'].isin(['moved', 'rotated', 'ate'])]
for turn, grp in q_events.groupby('turn'):
    if turn > 8: break
    moves = grp[grp['event'] == 'moved']
    rots = grp[grp['event'] == 'rotated']
    hexes = moves['dst_hex'].tolist()
    print(f"Turn {turn}: {len(rots)} rots, {len(moves)} moves → {hexes}  "
          f"(facing={queen.facing if turn == max(q_events.turn) else '?'})")

# Also check: is instruction cursor where we expect?
print(f"\nInstruction cursor: {queen.instructions.cursor}/{len(queen.instructions.rules)}")
print(f"Queen at: ⬡{queen.location}, facing: {queen.facing}")
print(f"Pawn at ⬡368: {any(p.location == 368 for p in pcr_squad.pieces if p != queen)}")
```

My best guess: the **`step()` method** calls `_tick_one` in a round-robin slot-based system where each slot gives the queen ONE `_tick_one` call (not a full budget loop). That would mean:

- Slot A: `_tick_one` → FWD(334→353), returns cost 1
- Slot B: `_tick_one` → ROT_R (free, continues), FWD(353→**352** if W, or **373** if still SW)

If the facing wasn't updated between slots (e.g., the rotation is logged but applied to a stale copy), she'd still be facing SW and go to 373.

The event log will confirm whether ROT_R actually fired before or after the second FORWARD.

In [ ]:
# What did the queen actually do each turn?
q_events = sim.df[sim.df['piece_id'] == queen_id].copy()
q_events = q_events[q_events['event'].isin(['moved', 'rotated', 'ate'])]
for turn, grp in q_events.groupby('turn'):
    if turn > 8: break
    moves = grp[grp['event'] == 'moved']
    rots = grp[grp['event'] == 'rotated']
    hexes = moves['dst_hex'].tolist()
    print(f"Turn {turn}: {len(rots)} rots, {len(moves)} moves → {hexes}  "
          f"(facing={queen.facing if turn == max(q_events.turn) else '?'})")

# Also check: is instruction cursor where we expect?
print(f"\nInstruction cursor: {queen.instructions.cursor}/{len(queen.instructions.rules)}")
print(f"Queen at: ⬡{queen.location}, facing: {queen.facing}")
print(f"Pawn at ⬡368: {any(p.location == 368 for p in pcr_squad.pieces if p != queen)}")


So a constraint of the route was that the pawns weren't supposed to be on it. But we can fix that one after we fix this one.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good data. The directions match perfectly, so the convention isn't the issue. The **real** problem is visible in the event log:

**Turn 4: 0 rots, 2 moves → [353, 373]**

She moved SW **twice** with no rotation between. The ROT_R instruction [11] was consumed but **didn't fire as a rotation**. Then the rotation shows up on turn 5 instead.

My best guess: `ROT_R`/`ROT_L` might not be in `_FREE` in the simulator, or `_try_forward` uses a different direction lookup. Can you check:

```python
from HexMagic.game.mechanics.simulator import _FREE, _COSTS_ONE

print(f"_FREE: {_FREE}")
print(f"_COSTS_ONE: {_COSTS_ONE}")
print(f"ROT_R in _FREE: {Instruction.ROT_R in _FREE}")
print(f"ROT_L in _FREE: {Instruction.ROT_L in _FREE}")
```

And also:

```python
??FoodSimulator.step
??FoodSimulator._try_forward
```

If ROT_R accidentally ended up in `_COSTS_ONE`, it would consume a budget slot without actually rotating the piece (since only HARVEST and GIVE are handled there), and wouldn't be logged as "rotated". That would explain everything — she burns budget on a no-op "rotation", stays facing SW, and takes an extra SW step.

In [ ]:
from HexMagic.game.mechanics.simulator import _FREE, _COSTS_ONE

print(f"_FREE: {_FREE}")
print(f"_COSTS_ONE: {_COSTS_ONE}")
print(f"ROT_R in _FREE: {Instruction.ROT_R in _FREE}")
print(f"ROT_L in _FREE: {Instruction.ROT_L in _FREE}")


In [ ]:
??FoodSimulator.step

In [ ]:
??FoodSimulator._try_forward

bug in the simulator?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Found it! Look at `_try_forward`:

```python
while auto_rots <= 5:
    d = HexPosition.directions()[piece.facing % 6]
    nbr = self.grid.hexposition_to_index(d, piece.location)
    passable = (0 <= nbr < len(self.elevations)
                and nbr not in self.grid.invalidRegion
                and self.elevations[nbr] >= 1)
    if self.countries is not None and passable:
        passable = self.countries[nbr] >= 0
    if not passable:
        piece.facing = (piece.facing - 1) % 6; auto_rots += 1; continue
```

**`_try_forward` has silent auto-rotation!** When the target hex is impassable, it rotates `facing - 1` and tries the next direction — without logging a rotation event.

So on Turn 4: the ROT_R correctly set facing=1 (W), targeting ⬡352. But `_try_forward` found **⬡352 is impassable** (elevation < 1 or country < 0), silently rotated back to facing=0 (SW), and moved to ⬡373 instead.

Let's confirm:

```python
print(f"⬡352 elevation: {myStuff.terr.elevations[352]}")
print(f"⬡352 country:   {myStuff.terr.fields.get('country', [None])[352] if myStuff.terr.fields.get('country') is not None else 'N/A'}")
print(f"⬡352 in invalid: {352 in myStuff.grid.invalidRegion}")
print(f"⬡352 food tier:  {fy.tiers[352]}")

# Check the whole route for passability
countries = myStuff.terr.fields.get('country')
for i, h in enumerate(pcr['route']):
    elev = myStuff.terr.elevations[h]
    ctry = countries[h] if countries is not None else -1
    ok = elev >= 1 and ctry >= 0
    if not ok:
        print(f"  ⚠️ route[{i}] ⬡{h}: elev={elev:.0f} country={ctry} {'BLOCKED' if not ok else ''}")
```

The PCR pathfinder probably doesn't check the `country` field, but the simulator does. That's the mismatch.

In [ ]:
print(f"⬡352 elevation: {myStuff.terr.elevations[352]}")
print(f"⬡352 country:   {myStuff.terr.fields.get('country', [None])[352] if myStuff.terr.fields.get('country') is not None else 'N/A'}")
print(f"⬡352 in invalid: {352 in myStuff.grid.invalidRegion}")
print(f"⬡352 food tier:  {fy.tiers[352]}")

# Check the whole route for passability
countries = myStuff.terr.fields.get('country')
for i, h in enumerate(pcr['route']):
    elev = myStuff.terr.elevations[h]
    ctry = countries[h] if countries is not None else -1
    ok = elev >= 1 and ctry >= 0
    if not ok:
        print(f"  ⚠️ route[{i}] ⬡{h}: elev={elev:.0f} country={ctry} {'BLOCKED' if not ok else ''}")


So the country check is a bad thing. we should be checking for elevation

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Agreed. The `_try_forward` method has two problems:

1. **Country check shouldn't gate movement** — elevation is what matters for passability
2. **Silent auto-rotation** — if the target is blocked, it quietly rotates facing and tries a different direction, which corrupts the instruction sequence

Here's the fix:

```python
@patch
def _try_forward(self: FoodSimulator, piece, occupied, slot=None):
    d = HexPosition.directions()[piece.facing % 6]
    nbr = self.grid.hexposition_to_index(d, piece.location)
    
    # Passability: only check elevation and grid bounds
    passable = (0 <= nbr < len(self.elevations)
                and nbr not in self.grid.invalidRegion
                and self.elevations[nbr] >= 1)
    
    if not passable:
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 0.0
    
    # Occupied check
    if nbr in occupied and nbr != piece.location:
        blocker = next((p for p in self.alive if p.location == nbr), None)
        if blocker:
            bump_cost = self._try_bump(piece, blocker, occupied, slot)
            if bump_cost is not None:
                return bump_cost
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 1.0
    
    cost = _move_cost(self.elevations, piece.location, nbr)
    src = piece.location
    occupied.discard(src); piece.location = nbr; occupied.add(nbr)
    self._log(piece, SimEventType.MOVED, slot=slot, src_hex=src, dst_hex=nbr, amount=cost)
    return cost
```

Changes:
- **Removed country check** — only elevation and grid bounds matter
- **Removed silent auto-rotation loop** — if the facing direction is blocked, just report BLOCKED and return 0. The instruction list controls facing, not the movement engine.

This way the queen follows her instructions exactly: rotate when told to, move forward when told to, and if blocked, just skip that turn instead of sneaking off in a different direction.

Did I put it in correctly?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, the `_try_forward` patch looks correct — no country check, no silent auto-rotation. ✅

But she's **still going off-route** — `334→353→373` on turn 4. The `_try_forward` fix isn't the issue here. Looking at the instruction list:

```
[10] FORWARD     ← moves 334→353 (correct)
[11] ROT_R       ← should fire next...
```

The problem is **timing within the turn budget**. She has high `move_strength` (3+), so in one slot she does FORWARD[10]→353, and in the next slot she should do ROT_R[11](free)+FORWARD[12]→352. But instead she goes to 373 (another SW step).

Let's check the move budget:

```python
print(f"Queen move_strength: {queen.move_strength}")
print(f"Queen move budget per turn: {float(queen.move_strength)}")

# Check elevation costs along the route
for i in range(len(pcr['route']) - 1):
    a, b = pcr['route'][i], pcr['route'][i+1]
    cost = _move_cost(myStuff.terr.elevations, a, b)
    print(f"  {a}→{b}: cost={cost:.2f} "
          f"(elev {myStuff.terr.elevations[a]:.0f}→{myStuff.terr.elevations[b]:.0f})")
```

Also let's see the **full** event log for turn 4 (including all event types, not just moved/rotated):

```python
t4 = sim.df[(sim.df['piece_id'] == queen_id) & (sim.df['turn'] == 4)]
print(t4[['turn', 'slot', 'event', 'src_hex', 'dst_hex', 'amount']].to_string(index=False))
```

This will tell us exactly what happened instruction-by-instruction on the critical turn.

In [ ]:
print(f"Queen move_strength: {queen.move_strength}")
print(f"Queen move budget per turn: {float(queen.move_strength)}")

# Check elevation costs along the route
for i in range(len(pcr['route']) - 1):
    a, b = pcr['route'][i], pcr['route'][i+1]
    cost = _move_cost(myStuff.terr.elevations, a, b)
    print(f"  {a}→{b}: cost={cost:.2f} "
          f"(elev {myStuff.terr.elevations[a]:.0f}→{myStuff.terr.elevations[b]:.0f})")


In [ ]:
t4 = sim.df[(sim.df['piece_id'] == queen_id) & (sim.df['turn'] == 4)]
print(t4[['turn', 'slot', 'event', 'src_hex', 'dst_hex', 'amount']].to_string(index=False))


NaN?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The NaN for `ate` is normal — `_eat_phase` doesn't pass a slot number, so it's None → NaN in pandas. Not a bug.

The **real** finding from this log: **Turn 4, slot 0: moved 334→353 (cost 2.0), slot 1: moved 353→373 (cost 1.0). Zero rotations.** The ROT_R at instruction[11] simply didn't fire between those two moves.

With budget=3 and cost 2.0 for the first move, she has 1.0 remaining. On slot 1, `_tick_one` should hit cursor=11 (ROT_R, free), rotate, advance to cursor=12 (FORWARD), and move W to 352. Instead she moved SW to 373.

Let's dump the full event-level detail:

```python
# Full event log for turns 3-5, ALL events
for t in range(3, 6):
    print(f"\n=== TURN {t} ===")
    t_events = sim.df[(sim.df['piece_id'] == queen_id) & (sim.df['turn'] == t)]
    print(t_events[['turn', 'slot', 'event', 'src_hex', 'dst_hex', 'amount']].to_string(index=False))

# Also verify: what does the cursor look like RIGHT before turn 4?
# Count total instructions consumed through turn 3
consumed = 0
for t in range(4):
    t_events = sim.df[(sim.df['piece_id'] == queen_id) & (sim.df['turn'] == t)]
    rots = len(t_events[t_events['event'] == 'rotated'])
    moves = len(t_events[t_events['event'] == 'moved'])
    consumed += rots + moves
    print(f"Turn {t}: {rots} rots + {moves} moves = {rots+moves} instructions (cumulative: {consumed})")

print(f"\nExpected cursor at start of turn 4: {consumed}")
print(f"Instruction[{consumed}] = {Instruction(queen.instructions.rules[consumed]).name}")
```

This should tell us if the cursor is at 10 (FORWARD) or somewhere else when turn 4 starts.

In [ ]:
# Full event log for turns 3-5, ALL events
for t in range(3, 6):
    print(f"\n=== TURN {t} ===")
    t_events = sim.df[(sim.df['piece_id'] == queen_id) & (sim.df['turn'] == t)]
    print(t_events[['turn', 'slot', 'event', 'src_hex', 'dst_hex', 'amount']].to_string(index=False))

# Also verify: what does the cursor look like RIGHT before turn 4?
# Count total instructions consumed through turn 3
consumed = 0
for t in range(4):
    t_events = sim.df[(sim.df['piece_id'] == queen_id) & (sim.df['turn'] == t)]
    rots = len(t_events[t_events['event'] == 'rotated'])
    moves = len(t_events[t_events['event'] == 'moved'])
    consumed += rots + moves
    print(f"Turn {t}: {rots} rots + {moves} moves = {rots+moves} instructions (cumulative: {consumed})")

print(f"\nExpected cursor at start of turn 4: {consumed}")
print(f"Instruction[{consumed}] = {Instruction(queen.instructions.rules[consumed]).name}")


?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Found it! The bug is in the interaction between `_tick_one` and `_try_forward`. Look at this sequence in `_tick_one`:

```python
cost = self._try_forward(piece, occupied, slot)  # ← MOVES the piece!
if cost > 0:
    if budget >= cost: il.cursor += 1; return Instruction.FORWARD, cost
    else: return None, 0.0  # ← piece already moved, but no budget deducted!
```

`_try_forward` **unconditionally moves the piece** before returning the cost. Then `_tick_one` checks `budget >= cost` — but it's too late, the piece already moved!

Evidence from Turn 0:
- Budget = 3.0
- Slot 0: ROT_L(free) + FWD 158→178 (cost 2.0) ✅ budget = 1.0
- Slot 1: FWD 178→197 (cost 2.0) — **piece moves**, but budget(1.0) < cost(2.0), so returns (None, 0). **Free move!** Cursor stays at [2].
- Slot 2: retries FWD[2] from 197→217 (cost 1.0) ✅ budget = 0

She got **3 moves on a budget of 3**, but the middle one was free. She's now permanently **one hex ahead** of where the cursor thinks she is. By instruction[10], she's past 353 and takes the extra SW step to 373 before ROT_R fires.

Fix — pass budget into `_try_forward` and check before moving:

```python
@patch
def _try_forward(self: FoodSimulator, piece, occupied, slot=None, budget=float('inf')):
    d = HexPosition.directions()[piece.facing % 6]
    nbr = self.grid.hexposition_to_index(d, piece.location)
    
    passable = (0 <= nbr < len(self.elevations)
                and nbr not in self.grid.invalidRegion
                and self.elevations[nbr] >= 1)
    
    if not passable:
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 0.0
    
    if nbr in occupied and nbr != piece.location:
        blocker = next((p for p in self.alive if p.location == nbr), None)
        if blocker:
            bump_cost = self._try_bump(piece, blocker, occupied, slot)
            if bump_cost is not None:
                return bump_cost
        self._log(piece, SimEventType.BLOCKED, slot=slot, dst_hex=nbr)
        return 0.0
    
    cost = _move_cost(self.elevations, piece.location, nbr)
    if cost > budget:
        return 0.0  # Can't afford — don't move!
    
    src = piece.location
    occupied.discard(src); piece.location = nbr; occupied.add(nbr)
    self._log(piece, SimEventType.MOVED, slot=slot, src_hex=src, dst_hex=nbr, amount=cost)
    return cost
```

And update the call in `_tick_one`:

```python
cost = self._try_forward(piece, occupied, slot, budget=budget)
if cost > 0:
    il.cursor += 1
    return Instruction.FORWARD, cost
return None, 0.0
```

can you give me a proper patch for _tick_one?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
@patch
def _tick_one(self: FoodSimulator, piece, budget, occupied, slot=None):
    il = piece.instructions
    if not il.rules: return None, 0.0
    while True:
        if il.cursor >= len(il.rules):
            if il.patrol: il.cursor = 0
            else: return None, 0.0
        instr = Instruction(il.rules[il.cursor])
        if instr in _FREE:
            piece.facing = ((piece.facing - 1) if instr == Instruction.ROT_L
                            else (piece.facing + 1)) % 6
            self._log(piece, SimEventType.ROTATED, slot=slot)
            il.cursor += 1; continue
        if instr in _COSTS_ONE:
            if budget < 1.0: return None, 0.0
            il.cursor += 1
            if instr == Instruction.HARVEST: self._do_harvest(piece, slot)
            elif instr == Instruction.GIVE: self._do_give(piece, slot)
            return instr, 1.0
        # FORWARD — pass budget so _try_forward won't move if we can't afford it
        if budget < 1.0: return None, 0.0
        cost = self._try_forward(piece, occupied, slot, budget=budget)
        if cost > 0:
            il.cursor += 1
            return Instruction.FORWARD, cost
        return None, 0.0
```

The only change from the original is passing `budget=budget` to `_try_forward`, and removing the `budget >= cost` check afterward (since `_try_forward` now handles that itself — it returns 0.0 if cost > budget, without moving the piece).

So this notebook is a little messy, but I am keeping a spare because of the big victory. I need to check this into git which removes prompts and clean it up in general. I need to prioritize what to keep. my thoughts are

1. apply_food_profile (though I need to pick the right one which I think is apply_food_profile_v2)
2. food_triage_v2
3. food_redistribute_adjacent/ food_redistribute_rescue
4. optimal_camp_layout
5. smart_placement_v2
6. pawn_covered_route
7. route_to_instructions (need to pick the right one)

I might restart SupplyRole/SupplyNode/SupplySchedule. I am assuming it ok to take them out.

Please always add thoughts about algorithims and how we troubleshot things. I don't like it when your comments go away, but I know you are great at summarizing.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Heads up: some earlier messages in our dialog were truncated due to context size, so I'm working from what's visible. That said, I have a good picture of the full arc.

Here's my recommended keep list with summary notes you can paste as markdown comments:

---

**1. `FOOD_PROFILE_V2` + `apply_food_profile_v2`** ✅ Keep

The rounded profile: `h=1.3, d=0.7, Q=7.0`. Found via Optuna optimization of `pcr_profile_objective` (100 trials). The key design rule: **two pawns with tier sum ≥ 7 feed the queen**. T5+T2 barely passes (export 7.7 vs Q=7.0), sum 6 fails (6.4). This tight margin is intentional game design. Queen/pawn diet ratio is 10× — pawns are cheap individually, quantity is the cost.

**2. `food_triage_v2`** ✅ Keep

The starvation-priority food redistribution. Key insight from debugging: the original triage was too aggressive — it would strip food from pawns that needed it. V2 respects a reserve floor (`FOOD_RESERVE_TURNS * diet`) before giving surplus to needier pieces.

**3. `food_redistribute_adjacent` / `food_redistribute_rescue`** ✅ Keep

Squad-level callbacks fired after each sim turn. `adjacent` handles normal sharing between nearby pieces. `rescue` is the emergency fallback for starving pieces. These plug into `FoodSimulator.callbacks` dict keyed by `squad.id`.

**4. `optimal_camp_layout`** ✅ Keep

LP-based placement using Pyomo. Maximizes surplus (production - consumption) for a settlement's pieces within a candidate radius. Outputs hex assignments and flow edges. Used by `smart_placement_v2` as the inner solver.

**5. `smart_placement_v2`** ✅ Keep

Wrapper that picks the best center hex from multiple candidates, then calls `optimal_camp_layout`. Evaluates ~5 candidates by surplus score. Places the queen at the highest-tier hex near center, with pawns fanning out on productive land.

**6. `pawn_covered_route`** ✅ Keep

The big win. BFS shortest path between settlements, then greedy set-cover to place pawns along the route so every hex has enough feeders within `cone_range`. Each pawn's export = `harvest * tier - diet`. The coverage constraint: overlapping pawn cones must sum to `queen_diet` at every route hex. Returns route, placements (with coverage windows), and gap list.

**7. `route_to_instructions`** ✅ Keep — use the **exact neighbor lookup** version

```python
def route_to_instructions(route, grid, start_facing):
    """Convert adjacent hex route into FORWARD/ROT instructions.
    
    Uses exact neighbor lookup (not angle approximation) to find
    which of the 6 directions connects each pair of adjacent hexes.
    """
```

The angle-based `_facing_toward` version had bugs where it picked the wrong neighbor on certain row parities. The correct version iterates `dirs[0..5]`, calls `grid.hexposition_to_index(dirs[d], route[i])`, and checks for exact match with `route[i+1]`.

---

**Safe to remove:**

- `SupplyRoute/SupplyNode/SupplySchedule` — yes, fine to restart these later. They were an earlier abstraction that the PCR approach superseded.
- The Optuna optimization code (`pcr_profile_objective`, study setup) — the result is captured in `FOOD_PROFILE_V2`, you don't need the search code.
- `simulate_queen_march` (the simple/ideal sim) — it was useful for validation but `FoodSimulator` is the real thing now.

**Key bugs we found (worth a comment block somewhere):**

1. **`_try_forward` moved before checking budget** — piece relocated unconditionally, then budget check failed, giving free moves. Fix: pass `budget` to `_try_forward`, check `cost > budget` *before* moving.
2. **`_try_forward` had silent auto-rotation** — when blocked, it rotated facing and retried, corrupting the instruction sequence. Fix: just return BLOCKED, let instructions control facing.
3. **`_try_forward` checked country field** — blocked movement across country boundaries even on passable terrain. Fix: only check elevation ≥ 1.
4. **Direction convention was fine** — `HexPosition.directions()` indices 0–5 match `piece.facing` 0–5 perfectly. The mismatch was in the budget bug, not the direction mapping.